# 🏛️ AI Legal & Government Document QA System

**Run in Google Colab** — fully self-contained notebook.

### Features
- 📄 Supports PDF (text + scanned), DOCX, TXT
- 🌐 Multilingual: English | Tamil | Hindi
- 🤖 Google Gemini 1.5 Flash LLM backend
- 🔍 BAAI/bge-m3 multilingual embeddings (1024-dim)
- 📝 Dynamic summary length based on document page count
- ✅ Answer verification & source citation

### Steps
1. Run **Cell 1** → Install dependencies
2. Run **Cell 2** → Set your Google API key
3. Run **Cells 3-13** → Write source files
4. Run **Cell 14** → Start the QA system


## Cell 1 — Install System & Python Dependencies

In [ ]:
# System packages (Poppler for PDF, Tesseract for OCR fallback)
!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr tesseract-ocr-tam tesseract-ocr-hin

# Core Python packages
!pip install -q \
    easyocr \
    faiss-cpu \
    sentence-transformers \
    pdfplumber \
    pdf2image \
    python-docx \
    langdetect \
    deep-translator \
    transformers \
    torch \
    numpy \
    python-dotenv \
    hf-transfer

# New Google Gemini SDK (replaces deprecated google-generativeai)
!pip install -q google-genai

print('✅ All dependencies installed.')


## Cell 2 — Set Your Google Gemini API Key

In [ ]:
import os

# ⚠️  Paste your Google AI Studio API key below
# Get a free key from: https://aistudio.google.com/app/apikey
GOOGLE_API_KEY = ""  # ← paste your key here

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
if GOOGLE_API_KEY:
    print('✅ API key set.')
else:
    print('⚠️  WARNING: No API key set. Gemini will not work.')


## Cells 3-13 — Write Source Files
Run all cells below to write the Python source files to the Colab environment.

### `config.py`

In [ ]:
%%writefile config.py
"""
config.py
---------
Central configuration for the AI Legal QA System.
All tunable parameters, model names, and system constants are defined here.
Never hardcode values inside individual modules — always import from config.
"""

import os
import logging
from pathlib import Path

# API key is set in Cell 2 via os.environ

# ─────────────────────────────────────────────
# PROJECT PATHS
# ─────────────────────────────────────────────
BASE_DIR = Path(__file__).parent.resolve()
TEMP_DIR = BASE_DIR / "temp"
LOG_DIR  = BASE_DIR / "logs"

# Ensure temp, log, and models directories exist at import time
TEMP_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

# ── Force ALL model caches into the project folder ──────────────────
# HuggingFace / Transformers
_HF_DIR = MODELS_DIR / "huggingface"
_HF_DIR.mkdir(exist_ok=True)
os.environ["HF_HOME"]              = str(_HF_DIR)
os.environ["HF_HUB_CACHE"]        = str(_HF_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"]   = str(_HF_DIR / "hub")
os.environ["HUGGINGFACE_HUB_CACHE"]= str(_HF_DIR / "hub")
# SentenceTransformers
os.environ["SENTENCE_TRANSFORMERS_HOME"] = str(MODELS_DIR / "sentence_transformers")
# Torch hub
os.environ["TORCH_HOME"]          = str(MODELS_DIR / "torch")
# Enable hf-transfer for maximum download speed (Rust-based, 5-10× faster)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ─────────────────────────────────────────────
# POPPLER PATH (Windows only)
# ─────────────────────────────────────────────
# Set this to the folder containing pdftoppm.exe and pdfinfo.exe.
# Example after extracting to C:\poppler:
#   POPPLER_PATH = r"C:\poppler\Library\bin"
# Leave as None to rely on system PATH instead.
POPPLER_PATH: str | None = None  # Colab: poppler installed via apt-get

# ─────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────
LOG_LEVEL       = logging.INFO
LOG_FORMAT      = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"
LOG_DATE_FORMAT = "%Y-%m-%d %H:%M:%S"
LOG_FILE        = LOG_DIR / "legal_qa.log"

# ─────────────────────────────────────────────
# SUPPORTED LANGUAGES
# ─────────────────────────────────────────────
SUPPORTED_LANGUAGES = {
    "en": "English",
    "ta": "Tamil",
    "hi": "Hindi",
}
INTERNAL_LANGUAGE = "en"   # All processing is done in English

# ─────────────────────────────────────────────
# SUPPORTED DOCUMENT TYPES
# ─────────────────────────────────────────────
SUPPORTED_EXTENSIONS = {".pdf", ".txt", ".docx"}

# ─────────────────────────────────────────────
# OCR SETTINGS
# ─────────────────────────────────────────────
# Primary OCR backend: "easyocr" | "tesseract"
OCR_BACKEND = "easyocr"

# EasyOCR CANNOT mix Tamil, Hindi, and English in a single reader.
# Each script group must be loaded separately:
#   Tamil  + English -> ["en", "ta"]
#   Hindi  + English -> ["en", "hi"]
#   English only     -> ["en"]
# The system auto-selects the correct reader based on detected language.
EASYOCR_LANG_ENGLISH = ["en"]
EASYOCR_LANG_TAMIL   = ["en", "ta"]
EASYOCR_LANG_HINDI   = ["en", "hi"]

# Tesseract fallback language string
TESSERACT_LANG = "eng+tam+hin"

# Minimum character count to consider a page as "text-based" (not scanned)
MIN_TEXT_CHARS_PER_PAGE = 30

# DPI for PDF-to-image conversion when OCR is needed
PDF_OCR_DPI = 300

# ─────────────────────────────────────────────
# LANGUAGE DETECTION
# ─────────────────────────────────────────────
# Library to use: "langdetect" | "langid"
LANG_DETECT_BACKEND = "langdetect"

# Minimum confidence for language detection (0-1); below this → default to English
LANG_DETECT_MIN_CONFIDENCE = 0.75

# ─────────────────────────────────────────────
# TRANSLATION
# ─────────────────────────────────────────────
# Backend: "indictrans2" | "googletrans" (fallback)
TRANSLATION_BACKEND = "googletrans"

# IndicTrans2 model checkpoint directory (downloaded at runtime if missing)
INDICTRANS2_MODEL_DIR = BASE_DIR / "models" / "indictrans2"

# Batch size for IndicTrans2 inference
INDICTRANS2_BATCH_SIZE = 8

# Maximum characters per translation chunk (to avoid OOM)
TRANSLATION_MAX_CHARS = 2000

# ─────────────────────────────────────────────
# EMBEDDING MODEL
# ─────────────────────────────────────────────
# Multilingual dense retrieval model
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"

# Fallback if BGE-M3 is unavailable
EMBEDDING_FALLBACK_MODEL = "intfloat/multilingual-e5-large"

# Maximum tokens per embedding call
EMBEDDING_MAX_TOKENS = 512

# Batch size for embedding generation
EMBEDDING_BATCH_SIZE = 16

# Normalize embeddings before FAISS indexing
NORMALIZE_EMBEDDINGS = True

# ─────────────────────────────────────────────
# CHUNKING
# ─────────────────────────────────────────────
# Structure-aware chunking patterns (regex anchors)
CHUNK_SECTION_PATTERNS = [
    r"(?m)^(CHAPTER\s+[IVXLCDM\d]+)",
    r"(?m)^(PART\s+[IVXLCDM\d]+)",
    r"(?m)^(SECTION\s+\d+[\.\d]*)",
    r"(?m)^(Section\s+\d+[\.\d]*)",
    r"(?m)^(ARTICLE\s+\d+[\.\d]*)",
    r"(?m)^(Article\s+\d+[\.\d]*)",
    r"(?m)^(CLAUSE\s+\d+[\.\d]*)",
    r"(?m)^(Clause\s+\d+[\.\d]*)",
    r"(?m)^(RULE\s+\d+[\.\d]*)",
    r"(?m)^(Rule\s+\d+[\.\d]*)",
    r"(?m)^(GO\s+(?:No\.?\s*)?\d+)",
    r"(?m)^(G\.O\.\s+(?:No\.?\s*)?\d+)",
    r"(?m)^(\d+\.\s+[A-Z])",          # Numbered top-level items
    r"(?m)^(\(\d+\)\s+[A-Z])",        # Parenthesised items
]

# Soft max / min tokens per chunk (structure takes priority)
CHUNK_SOFT_MAX_TOKENS = 400
CHUNK_SOFT_MIN_TOKENS = 50

# Overlap between adjacent chunks (in tokens) for context continuity
CHUNK_OVERLAP_TOKENS  = 50

# ─────────────────────────────────────────────
# FAISS / RETRIEVAL
# ─────────────────────────────────────────────
# FAISS index type: "flat_l2" | "flat_ip" | "ivf"
FAISS_INDEX_TYPE = "flat_ip"   # Inner Product (cosine after normalisation)

# Number of top-k chunks to retrieve
RETRIEVAL_TOP_K = 8

# Minimum similarity score (0-1) for a chunk to be considered relevant
RETRIEVAL_MIN_SCORE = -1.0

# ─────────────────────────────────────────────
# LLM / GENERATION
# ─────────────────────────────────────────────
# LLM & RAG CONFIGURATION
# ─────────────────────────────────────────────

# LLM backend selection
LLM_BACKEND = "huggingface"

# HuggingFace model for reader (must be instruction-tuned)
HF_READER_MODEL = "google/flan-t5-base"

# OpenAI model name (used only if LLM_BACKEND == "openai")
OPENAI_MODEL = "gpt-4o-mini"

# Google Gemini model (used only if LLM_BACKEND == "google")
GEMINI_MODEL = "gemini-3.5-flash"

# Maximum new tokens to generate in the answer
LLM_MAX_NEW_TOKENS = 512

# Temperature — keep low for factual legal answers
LLM_TEMPERATURE = 0.0

# ─────────────────────────────────────────────
# VERIFICATION
# ─────────────────────────────────────────────
# Minimum overlap ratio (0-1) between answer tokens and retrieved context
VERIFICATION_MIN_OVERLAP = 0.40

# Maximum retry attempts before returning "not found"
VERIFICATION_MAX_RETRIES = 1

# Confidence threshold (0-100) below which answer is rejected
VERIFICATION_MIN_CONFIDENCE = 40.0

# ─────────────────────────────────────────────
# PROMPT TEMPLATES
# ─────────────────────────────────────────────
RAG_SYSTEM_PROMPT = (
    "You are a strict legal document assistant. "
    "Answer ONLY using information from the provided document excerpts. "
    "If the user asks for a summary, provide a detailed summary of between {min_lines} and {max_lines} lines. "
    "If the answer is not present in the excerpts, respond exactly with: "
    "'Information not found in the uploaded document.' "
    "Do NOT use any external knowledge. Do NOT hallucinate."
)

RAG_USER_PROMPT_TEMPLATE = (
    "Document Excerpts:\n"
    "{context}\n\n"
    "Question: {question}\n\n"
    "Answer (cite the section/clause/GO number if mentioned):"
)

NOT_FOUND_RESPONSE = "Information not found in the uploaded document."


### `utils.py`

In [ ]:
%%writefile utils.py
"""
utils.py
--------
Shared utility functions used across all modules.
Provides: logging setup, file validation, text helpers, timer, colour output.
"""

import re
import sys
import time
import logging
import unicodedata
from pathlib import Path
from typing import Optional

import config

# ─────────────────────────────────────────────
# LOGGING SETUP
# ─────────────────────────────────────────────

def setup_logging(name: str = "legal_qa") -> logging.Logger:
    """
    Configure and return a named logger that writes to both the console
    and a rotating log file defined in config.

    Args:
        name: Logger name (typically the module __name__).

    Returns:
        Configured logging.Logger instance.
    """
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger  # Already configured — avoid duplicate handlers

    logger.setLevel(config.LOG_LEVEL)
    formatter = logging.Formatter(
        fmt=config.LOG_FORMAT,
        datefmt=config.LOG_DATE_FORMAT,
    )

    # Console handler
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(config.LOG_LEVEL)
    ch.setFormatter(formatter)
    logger.addHandler(ch)

    # File handler
    try:
        fh = logging.FileHandler(config.LOG_FILE, encoding="utf-8")
        fh.setLevel(config.LOG_LEVEL)
        fh.setFormatter(formatter)
        logger.addHandler(fh)
    except OSError as exc:
        logger.warning("Could not open log file %s: %s", config.LOG_FILE, exc)

    return logger


# Module-level logger
_log = setup_logging(__name__)

# ─────────────────────────────────────────────
# CONSOLE COLOUR OUTPUT
# ─────────────────────────────────────────────

ANSI = {
    "reset":  "\033[0m",
    "bold":   "\033[1m",
    "green":  "\033[92m",
    "yellow": "\033[93m",
    "red":    "\033[91m",
    "cyan":   "\033[96m",
    "blue":   "\033[94m",
    "magenta":"\033[95m",
    "white":  "\033[97m",
    "dim":    "\033[2m",
}

# Detect whether the terminal supports ANSI codes
_SUPPORTS_COLOUR = hasattr(sys.stdout, "isatty") and sys.stdout.isatty()


def cprint(text: str, colour: str = "white", bold: bool = False) -> None:
    """
    Print coloured text to stdout if the terminal supports it.

    Args:
        text:   The string to print.
        colour: Key from ANSI dict (e.g. 'green', 'red').
        bold:   Whether to apply bold formatting.
    """
    if _SUPPORTS_COLOUR:
        prefix = (ANSI.get("bold", "") if bold else "") + ANSI.get(colour, "")
        print(f"{prefix}{text}{ANSI['reset']}")
    else:
        print(text)


def print_separator(char: str = "─", width: int = 70, colour: str = "dim") -> None:
    """Print a horizontal separator line."""
    cprint(char * width, colour=colour)


def print_section_header(title: str) -> None:
    """Print a styled section header."""
    print_separator()
    cprint(f"  {title}", colour="cyan", bold=True)
    print_separator()

# ─────────────────────────────────────────────
# FILE VALIDATION
# ─────────────────────────────────────────────

def validate_file(path: str) -> Path:
    """
    Validate that the given path points to an existing, readable file with
    a supported extension.

    Args:
        path: String path to the document file.

    Returns:
        Resolved Path object.

    Raises:
        FileNotFoundError: If the file does not exist.
        ValueError: If the file extension is not supported.
    """
    p = Path(path).resolve()
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")
    if not p.is_file():
        raise ValueError(f"Path is not a file: {p}")
    if p.suffix.lower() not in config.SUPPORTED_EXTENSIONS:
        raise ValueError(
            f"Unsupported file type '{p.suffix}'. "
            f"Supported: {', '.join(config.SUPPORTED_EXTENSIONS)}"
        )
    _log.debug("File validated: %s", p)
    return p


def get_file_size_mb(path: Path) -> float:
    """Return file size in megabytes."""
    return path.stat().st_size / (1024 * 1024)

# ─────────────────────────────────────────────
# TEXT UTILITIES
# ─────────────────────────────────────────────

def normalize_unicode(text: str) -> str:
    """
    Normalize Unicode to NFC form and strip zero-width / control characters.

    Args:
        text: Raw input string.

    Returns:
        NFC-normalized, control-char-stripped string.
    """
    text = unicodedata.normalize("NFC", text)
    # Remove control characters except newline (\n), carriage return (\r), tab (\t)
    text = "".join(
        ch for ch in text
        if unicodedata.category(ch)[0] != "C" or ch in "\n\r\t"
    )
    return text


def collapse_whitespace(text: str) -> str:
    """
    Collapse multiple consecutive spaces/tabs into a single space.
    Collapse more than two consecutive newlines into two.

    Args:
        text: Input string.

    Returns:
        Whitespace-normalised string.
    """
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def count_tokens_approx(text: str) -> int:
    """
    Approximate token count using word split (1 token ≈ 1 word for English).
    For multilingual text a rough estimate is sufficient.

    Args:
        text: Input string.

    Returns:
        Integer approximation of token count.
    """
    return len(text.split())


def truncate_text(text: str, max_chars: int) -> str:
    """
    Truncate text to at most `max_chars` characters, ending at a word boundary.

    Args:
        text:      Input string.
        max_chars: Maximum allowed characters.

    Returns:
        Truncated string.
    """
    if len(text) <= max_chars:
        return text
    truncated = text[:max_chars]
    last_space = truncated.rfind(" ")
    return truncated[:last_space] if last_space > 0 else truncated


def extract_go_numbers(text: str) -> list[str]:
    """
    Extract Government Order (GO) numbers from text using common GO reference
    patterns found in Indian government documents.

    Args:
        text: Document text.

    Returns:
        List of unique GO number strings found.
    """
    patterns = [
        r"G\.O\.\s*(?:Ms\.|Rt\.|No\.?)?\s*\d+",
        r"GO\s*(?:Ms\.|Rt\.|No\.?)?\s*\d+",
        r"Government Order\s+(?:No\.?)?\s*\d+",
    ]
    found: set[str] = set()
    for pat in patterns:
        for match in re.finditer(pat, text, re.IGNORECASE):
            found.add(match.group().strip())
    return sorted(found)

# ─────────────────────────────────────────────
# PERFORMANCE TIMING
# ─────────────────────────────────────────────

class Timer:
    """
    Context manager for wall-clock timing of code blocks.

    Usage:
        with Timer("Embedding generation") as t:
            ...
        print(t.elapsed_seconds)
    """

    def __init__(self, label: str = "", logger: Optional[logging.Logger] = None) -> None:
        self.label            = label
        self._logger          = logger or _log
        self.elapsed_seconds  = 0.0

    def __enter__(self) -> "Timer":
        self._start = time.perf_counter()
        return self

    def __exit__(self, *_) -> None:
        self.elapsed_seconds = time.perf_counter() - self._start
        if self.label:
            self._logger.debug(
                "%s completed in %.2fs", self.label, self.elapsed_seconds
            )


# ─────────────────────────────────────────────
# INTERACTIVE PROMPT HELPERS
# ─────────────────────────────────────────────

def prompt_user(prompt_text: str, default: str = "") -> str:
    """
    Display a styled prompt and return the user's input.
    Falls back to `default` if user presses Enter without input.

    Args:
        prompt_text: The prompt string to display.
        default:     Default value if input is empty.

    Returns:
        User's input string (stripped), or `default`.
    """
    if _SUPPORTS_COLOUR:
        display = f"{ANSI['bold']}{ANSI['blue']}▶ {prompt_text}{ANSI['reset']} "
    else:
        display = f"▶ {prompt_text} "

    try:
        value = input(display).strip()
    except (EOFError, KeyboardInterrupt):
        print()
        return default

    return value if value else default


def confirm(prompt_text: str) -> bool:
    """
    Ask a yes/no confirmation question.

    Args:
        prompt_text: Question to display.

    Returns:
        True if user typed 'y' or 'yes', False otherwise.
    """
    answer = prompt_user(f"{prompt_text} [y/N]:", default="n").lower()
    return answer in {"y", "yes"}


### `ocr.py`

In [ ]:
%%writefile ocr.py
"""
ocr.py
------
Optical Character Recognition module.

Responsibilities:
  1. Detect whether a PDF page is scanned (image-based) or text-based.
  2. Extract text from text-based PDFs using pdfplumber.
  3. Extract text from scanned PDFs / images using EasyOCR or Tesseract OCR.
  4. Extract text from DOCX and plain-text files.
  5. Preserve structure: paragraphs, section headings, numbering.

Design decision: EasyOCR is preferred because it natively supports Tamil and
Hindi without separate language pack installation. Tesseract is the fallback.
"""

import io
import logging
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

# Monkey patch for EasyOCR 1.6.2 compatibility with modern Pillow (>=10.0.0)
import PIL.Image
if not hasattr(PIL.Image, "ANTIALIAS"):
    PIL.Image.ANTIALIAS = getattr(PIL.Image, "LANCZOS", getattr(PIL.Image.Resampling, "LANCZOS", 1))

import config
from utils import setup_logging, Timer

logger = setup_logging(__name__)


# ─────────────────────────────────────────────
# DATA MODEL
# ─────────────────────────────────────────────

@dataclass
class PageResult:
    """
    Holds the OCR / extraction result for a single document page.

    Attributes:
        page_number:  1-based page index.
        text:         Extracted text content.
        is_ocr:       True if OCR was used (page was scanned).
        confidence:   Mean OCR confidence (0-100) if applicable, else None.
    """
    page_number: int
    text:        str
    is_ocr:      bool = False
    confidence:  Optional[float] = None


@dataclass
class DocumentOCRResult:
    """
    Aggregated extraction result for an entire document.

    Attributes:
        file_path:    Absolute path to the source file.
        file_type:    Extension: 'pdf', 'txt', 'docx'.
        pages:        List of PageResult objects.
        used_ocr:     True if any page required OCR.
        raw_text:     Full concatenated text (all pages joined).
    """
    file_path: Path
    file_type: str
    pages:     list[PageResult] = field(default_factory=list)
    used_ocr:  bool = False

    @property
    def raw_text(self) -> str:
        """Return full document text by joining all pages."""
        return "\n\n".join(p.text for p in self.pages if p.text.strip())

    @property
    def total_pages(self) -> int:
        return len(self.pages)


# ─────────────────────────────────────────────
# LAZY IMPORT HELPERS
# ─────────────────────────────────────────────

def _import_pdfplumber():
    """Lazy import pdfplumber to avoid hard dependency at module load."""
    try:
        import pdfplumber
        return pdfplumber
    except ImportError as e:
        raise ImportError(
            "pdfplumber is required for PDF processing.\n"
            "Install with: pip install pdfplumber"
        ) from e


def _import_easyocr():
    """Lazy import EasyOCR."""
    try:
        import easyocr
        return easyocr
    except ImportError as e:
        raise ImportError(
            "easyocr is required for scanned PDF processing.\n"
            "Install with: pip install easyocr"
        ) from e


def _import_pytesseract():
    """Lazy import pytesseract."""
    try:
        import pytesseract
        return pytesseract
    except ImportError as e:
        raise ImportError(
            "pytesseract is required for Tesseract OCR fallback.\n"
            "Install with: pip install pytesseract"
        ) from e


def _import_pdf2image():
    """Lazy import pdf2image for PDF-to-image conversion."""
    try:
        from pdf2image import convert_from_path
        return convert_from_path
    except ImportError as e:
        raise ImportError(
            "pdf2image is required for scanned PDF processing.\n"
            "Install with: pip install pdf2image\n"
            "Also install Poppler: https://github.com/oschwartz10612/poppler-windows/releases/"
        ) from e


def _import_docx():
    """Lazy import python-docx."""
    try:
        import docx
        return docx
    except ImportError as e:
        raise ImportError(
            "python-docx is required for DOCX processing.\n"
            "Install with: pip install python-docx"
        ) from e


# ─────────────────────────────────────────────
# SCANNED PAGE DETECTION
# ─────────────────────────────────────────────

def _is_page_scanned(page) -> bool:
    """
    Determine if a pdfplumber Page object is scanned (image-based).
    Heuristic: if extracted text has fewer than MIN_TEXT_CHARS_PER_PAGE
    printable characters, treat as scanned.

    Args:
        page: pdfplumber.Page object.

    Returns:
        True if the page is likely scanned.
    """
    text = page.extract_text() or ""
    printable = sum(1 for c in text if c.isprintable() and not c.isspace())
    is_scanned = printable < config.MIN_TEXT_CHARS_PER_PAGE
    logger.debug(
        "Page %d — printable chars: %d → %s",
        page.page_number,
        printable,
        "SCANNED" if is_scanned else "TEXT",
    )
    return is_scanned


# ─────────────────────────────────────────────
# EASYOCR ENGINE
# ─────────────────────────────────────────────

class EasyOCREngine:
    """
    Wrapper around EasyOCR for multilingual text extraction.

    EasyOCR restriction: Tamil, Hindi, and English cannot all be loaded in
    one Reader — they belong to different script groups.

    Solution: three separate cached readers, selected by language code:
        'en' → English-only reader
        'ta' → Tamil + English reader
        'hi' → Hindi  + English reader
    """

    _readers: dict = {}   # Cache: lang_key -> reader instance

    @classmethod
    def _get_reader(cls, lang: str = "en"):
        """
        Return (and cache) the appropriate EasyOCR Reader for the language.

        Args:
            lang: ISO code — 'en', 'ta', or 'hi'.

        Returns:
            easyocr.Reader instance.
        """
        easyocr = _import_easyocr()

        if lang == "ta":
            key   = "ta"
            langs = config.EASYOCR_LANG_TAMIL
        elif lang == "hi":
            key   = "hi"
            langs = config.EASYOCR_LANG_HINDI
        else:
            key   = "en"
            langs = config.EASYOCR_LANG_ENGLISH

        if key not in cls._readers:
            logger.info("Initialising EasyOCR reader for: %s", langs)
            easyocr_model_dir = str(config.MODELS_DIR / "easyocr")

            try:
                cls._readers[key] = easyocr.Reader(
                    langs, gpu=False,
                    model_storage_directory=easyocr_model_dir,
                    quantize=False,
                )
                logger.info("EasyOCR reader [%s] ready.", key)
            except Exception as exc:
                # Model file is corrupted or version-mismatched — delete and retry once
                import glob as _glob, os as _os
                deleted = False
                for bad_file in _glob.glob(_os.path.join(easyocr_model_dir, f"{key}*")) + \
                                _glob.glob(_os.path.join(easyocr_model_dir, "tamil*")):
                    try:
                        _os.remove(bad_file)
                        logger.info("Deleted corrupted model file: %s", bad_file)
                        deleted = True
                    except OSError:
                        pass

                if deleted:
                    logger.info("Retrying EasyOCR reader [%s] after cleanup…", key)
                    try:
                        cls._readers[key] = easyocr.Reader(
                            langs, gpu=False,
                            model_storage_directory=easyocr_model_dir,
                            quantize=False,
                        )
                        logger.info("EasyOCR reader [%s] ready after retry.", key)
                        return cls._readers[key]
                    except Exception as exc2:
                        exc = exc2

                logger.warning(
                    "EasyOCR reader [%s] failed (%s). "
                    "Falling back to English-only reader.", key, exc,
                )
                if "en" not in cls._readers:
                    logger.info("Loading English-only EasyOCR reader as fallback.")
                    cls._readers["en"] = easyocr.Reader(
                        config.EASYOCR_LANG_ENGLISH, gpu=False,
                        model_storage_directory=easyocr_model_dir,
                        quantize=False,
                    )
                cls._readers[key] = cls._readers["en"]
                logger.info("Fallback English reader will be used for [%s].", key)



        return cls._readers[key]

    @classmethod
    def extract_from_image(cls, image, lang: str = "en") -> tuple[str, float]:
        """
        Run EasyOCR on a PIL Image and return (text, mean_confidence).

        Args:
            image: PIL.Image object.
            lang:  ISO language code of the document ('en', 'ta', 'hi').

        Returns:
            Tuple of (extracted_text, confidence_percent).
        """
        import numpy as np
        reader    = cls._get_reader(lang)
        img_array = np.array(image)
        results   = reader.readtext(img_array, detail=1, paragraph=True)

        lines:       list[str]   = []
        confidences: list[float] = []
        for result in results:
            if len(result) == 3:
                _, text, conf = result
            elif len(result) == 2:
                _, text = result
                conf = 1.0
            else:
                continue
            if text.strip():
                lines.append(text.strip())
                confidences.append(conf * 100)

        full_text = "\n".join(lines)
        mean_conf = sum(confidences) / len(confidences) if confidences else 0.0
        return full_text, mean_conf


# ─────────────────────────────────────────────
# TESSERACT ENGINE (FALLBACK)
# ─────────────────────────────────────────────

class TesseractEngine:
    """Wrapper around pytesseract for OCR extraction."""

    @staticmethod
    def extract_from_image(image) -> tuple[str, float]:
        """
        Run Tesseract OCR on a PIL Image.

        Args:
            image: PIL.Image object.

        Returns:
            Tuple of (extracted_text, mean_confidence).
        """
        pytesseract = _import_pytesseract()
        data = pytesseract.image_to_data(
            image,
            lang=config.TESSERACT_LANG,
            output_type=pytesseract.Output.DICT,
        )
        words       = data["text"]
        confs       = data["conf"]
        valid_words = [
            w for w, c in zip(words, confs)
            if isinstance(c, (int, float)) and int(c) >= 0 and w.strip()
        ]
        valid_confs = [
            int(c) for c in confs
            if isinstance(c, (int, float)) and int(c) >= 0
        ]
        text     = " ".join(valid_words)
        mean_conf = sum(valid_confs) / len(valid_confs) if valid_confs else 0.0
        return text, mean_conf


# ─────────────────────────────────────────────
# MAIN OCR PROCESSOR
# ─────────────────────────────────────────────

class OCRProcessor:
    """
    Main entry point for document text extraction.

    Supports PDF (text-based and scanned), TXT, and DOCX.
    Automatically selects EasyOCR or Tesseract based on config.OCR_BACKEND.
    """

    def __init__(self) -> None:
        self._ocr_engine = (
            EasyOCREngine()
            if config.OCR_BACKEND == "easyocr"
            else TesseractEngine()
        )
        logger.info("OCRProcessor initialised with backend: %s", config.OCR_BACKEND)

    # ── Public API ──────────────────────────────────────────────────────────

    def process(self, file_path: Path) -> DocumentOCRResult:
        """
        Process a document file and return its extracted text.

        Args:
            file_path: Validated Path object to the document.

        Returns:
            DocumentOCRResult containing per-page results and combined text.

        Raises:
            ValueError: If the file type is unsupported.
        """
        suffix = file_path.suffix.lower()
        logger.info("Processing file: %s (type: %s)", file_path.name, suffix)

        with Timer(f"Extraction [{file_path.name}]", logger):
            if suffix == ".pdf":
                return self._process_pdf(file_path)
            elif suffix == ".txt":
                return self._process_txt(file_path)
            elif suffix == ".docx":
                return self._process_docx(file_path)
            else:
                raise ValueError(f"Unsupported file type: {suffix}")

    # ── PDF ─────────────────────────────────────────────────────────────────

    def _process_pdf(self, file_path: Path) -> DocumentOCRResult:
        """Extract text from PDF, using OCR for scanned pages."""
        pdfplumber = _import_pdfplumber()
        result = DocumentOCRResult(file_path=file_path, file_type="pdf")

        with pdfplumber.open(file_path) as pdf:
            total = len(pdf.pages)
            logger.info("PDF has %d page(s).", total)

            # First pass: identify scanned pages
            scanned_indices = [
                i for i, pg in enumerate(pdf.pages)
                if _is_page_scanned(pg)
            ]
            needs_ocr = bool(scanned_indices)
            logger.info(
                "Scanned pages detected: %s",
                scanned_indices if scanned_indices else "none",
            )

            # Detect document language from any available text
            # (used to pick the correct EasyOCR reader)
            ocr_lang = _detect_ocr_language(pdf)
            logger.info("OCR language selected: %s", ocr_lang)

            if needs_ocr:
                # Convert entire PDF to images for OCR pages
                convert_from_path = _import_pdf2image()
                logger.info(
                    "Converting PDF to images at %d DPI for OCR…", config.PDF_OCR_DPI
                )
                images = convert_from_path(
                    str(file_path),
                    dpi=config.PDF_OCR_DPI,
                    poppler_path=config.POPPLER_PATH,  # None = use system PATH
                )
            else:
                images = []

            for i, page in enumerate(pdf.pages):
                page_num = i + 1
                if i in scanned_indices:
                    text, conf = self._ocr_image(images[i], lang=ocr_lang)
                    result.pages.append(
                        PageResult(
                            page_number=page_num,
                            text=text,
                            is_ocr=True,
                            confidence=round(conf, 2),
                        )
                    )
                    result.used_ocr = True
                else:
                    raw = page.extract_text(
                        x_tolerance=3, y_tolerance=3
                    ) or ""
                    result.pages.append(
                        PageResult(
                            page_number=page_num,
                            text=raw,
                            is_ocr=False,
                        )
                    )

        logger.info(
            "PDF extraction complete. Pages: %d, OCR used: %s",
            result.total_pages,
            result.used_ocr,
        )
        return result

    # ── TXT ─────────────────────────────────────────────────────────────────

    def _process_txt(self, file_path: Path) -> DocumentOCRResult:
        """Read plain-text file with automatic encoding detection."""
        result = DocumentOCRResult(file_path=file_path, file_type="txt")
        for encoding in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
            try:
                text = file_path.read_text(encoding=encoding)
                result.pages.append(PageResult(page_number=1, text=text))
                logger.info(
                    "TXT file read with encoding '%s', %d chars.",
                    encoding, len(text),
                )
                return result
            except UnicodeDecodeError:
                continue
        raise UnicodeDecodeError(
            "utf-8", b"", 0, 1,
            f"Could not decode {file_path.name} with any supported encoding."
        )

    # ── DOCX ────────────────────────────────────────────────────────────────

    def _process_docx(self, file_path: Path) -> DocumentOCRResult:
        """
        Extract text from DOCX preserving paragraph structure.
        Each paragraph becomes a line; tables are extracted row-by-row.
        """
        docx = _import_docx()
        result = DocumentOCRResult(file_path=file_path, file_type="docx")
        doc = docx.Document(str(file_path))

        lines: list[str] = []
        for para in doc.paragraphs:
            if para.text.strip():
                lines.append(para.text)

        # Also extract table contents
        for table in doc.tables:
            for row in table.rows:
                row_text = " | ".join(
                    cell.text.strip()
                    for cell in row.cells
                    if cell.text.strip()
                )
                if row_text:
                    lines.append(row_text)

        full_text = "\n".join(lines)
        result.pages.append(PageResult(page_number=1, text=full_text))
        logger.info(
            "DOCX extraction complete. Paragraphs: %d, total chars: %d",
            len(lines), len(full_text),
        )
        return result

    # ── OCR DISPATCH ─────────────────────────────────────────────────────────

    def _ocr_image(self, image, lang: str = "en") -> tuple[str, float]:
        """
        Dispatch an image to the configured OCR engine.

        Args:
            image: PIL.Image object.
            lang:  Detected document language ('en', 'ta', 'hi').

        Returns:
            (text, confidence_percent) tuple.
        """
        if config.OCR_BACKEND == "easyocr":
            return EasyOCREngine.extract_from_image(image, lang=lang)
        else:
            return TesseractEngine.extract_from_image(image)


# ─────────────────────────────────────────────
# LANGUAGE PRE-DETECTION FOR OCR READER SELECTION
# ─────────────────────────────────────────────

def _detect_ocr_language(pdf) -> str:
    """
    Detect the dominant language of a PDF before full OCR.
    Samples text from the first available text-based page.
    Falls back to 'ta' (Tamil) if no text is found — safe default
    because Tamil+English reader handles English text fine too.

    Args:
        pdf: Open pdfplumber PDF object.

    Returns:
        ISO language code: 'en', 'ta', or 'hi'.
    """
    sample_text = ""
    for page in pdf.pages[:3]:   # Check first 3 pages
        t = page.extract_text() or ""
        if len(t.strip()) > 20:
            sample_text = t
            break

    if not sample_text.strip():
        # No extractable text — assume Tamil (covers Tamil+English docs)
        logger.debug("No text sample for language detection — defaulting to 'ta'.")
        return "ta"

    try:
        from langdetect import detect, DetectorFactory
        DetectorFactory.seed = 42
        code = detect(sample_text)
        if code == "ta":
            return "ta"
        elif code == "hi":
            return "hi"
        else:
            return "en"
    except Exception:
        return "ta"  # Safe fallback


### `translator.py`

In [ ]:
%%writefile translator.py
"""
translator.py
-------------
Multilingual translation module.

Responsibilities:
  1. Detect the language of a given text string.
  2. Translate Tamil / Hindi → English (for document text and user queries).
  3. Translate English → Tamil / Hindi (for answer localisation).
  4. Pass English text through unchanged.

Translation backends (in priority order):
  1. IndicTrans2  — best quality for Indian languages (preferred)
  2. deep-translator / googletrans — lightweight fallback

Language detection backend:
  - langdetect (default) with langid fallback

Design decision: Translation is done sentence-batch-by-sentence-batch to stay
within token/character limits and to avoid OOM for large documents.
"""

import re
import logging
from typing import Optional

import config
from utils import setup_logging, truncate_text, Timer

logger = setup_logging(__name__)

# ─────────────────────────────────────────────
# LANGUAGE DETECTION
# ─────────────────────────────────────────────

def detect_language(text: str) -> str:
    """
    Detect the dominant language of the provided text.

    Detection pipeline:
        1. Try langdetect.
        2. If confidence is below threshold, fall back to langid.
        3. If the detected code is not in SUPPORTED_LANGUAGES, default to 'en'.

    Args:
        text: Input text (at least a few sentences for accuracy).

    Returns:
        ISO 639-1 language code: 'en', 'ta', or 'hi'.
        Defaults to 'en' on failure.
    """
    sample = text[:2000].strip()   # Use up to 2000 chars for speed
    if not sample:
        logger.warning("Empty text passed to detect_language — defaulting to 'en'.")
        return "en"

    lang_code = _detect_with_langdetect(sample)
    if lang_code is None:
        lang_code = _detect_with_langid(sample)

    if lang_code not in config.SUPPORTED_LANGUAGES:
        logger.info(
            "Detected '%s' not in SUPPORTED_LANGUAGES — defaulting to 'en'.", lang_code
        )
        lang_code = "en"

    logger.info(
        "Language detected: %s (%s)",
        lang_code,
        config.SUPPORTED_LANGUAGES.get(lang_code, "unknown"),
    )
    return lang_code


def _detect_with_langdetect(text: str) -> Optional[str]:
    """Try langdetect library."""
    try:
        from langdetect import detect, DetectorFactory, LangDetectException
        DetectorFactory.seed = 42  # Deterministic output
        code = detect(text)
        # langdetect may return 'ta' for Tamil, 'hi' for Hindi
        return code
    except Exception as exc:
        logger.debug("langdetect failed: %s", exc)
        return None


def _detect_with_langid(text: str) -> Optional[str]:
    """Try langid library as fallback."""
    try:
        import langid
        code, _ = langid.classify(text)
        return code
    except Exception as exc:
        logger.debug("langid failed: %s", exc)
        return None


# ─────────────────────────────────────────────
# TRANSLATION ENGINE — IndicTrans2
# ─────────────────────────────────────────────

class IndicTrans2Engine:
    """
    Translation engine using IndicTrans2 (AI4Bharat).
    Lazily loads models on first use.

    IndicTrans2 model directions:
        - indic-en : Tamil/Hindi → English
        - en-indic : English → Tamil/Hindi
    """

    # Flores-200 language codes used by IndicTrans2
    _FLORES_CODES = {
        "en": "eng_Latn",
        "ta": "tam_Taml",
        "hi": "hin_Deva",
    }

    def __init__(self) -> None:
        self._en2indic_pipeline = None
        self._indic2en_pipeline = None
        logger.info("IndicTrans2Engine created (models load on first use).")

    # ── Pipeline loader ─────────────────────────────────────────────────────

    def _load_pipeline(self, direction: str):
        """
        Load an IndicTrans2 inference pipeline.

        Args:
            direction: 'indic-en' or 'en-indic'

        Returns:
            IndicTrans2 pipeline object.
        """
        try:
            from IndicTransToolkit import IndicTransTokenizer, IndicTransliterator
            from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
            import torch

            model_name = (
                "ai4bharat/indictrans2-indic-en-1B"
                if direction == "indic-en"
                else "ai4bharat/indictrans2-en-indic-1B"
            )
            logger.info("Loading IndicTrans2 model: %s", model_name)
            tokenizer = AutoTokenizer.from_pretrained(
                model_name, trust_remote_code=True
            )
            model = AutoModelForSeq2SeqLM.from_pretrained(
                model_name, trust_remote_code=True
            )
            model.eval()
            device = "cuda" if torch.cuda.is_available() else "cpu"
            model = model.to(device)
            logger.info("IndicTrans2 model '%s' loaded on %s.", model_name, device)
            return tokenizer, model, device

        except ImportError as exc:
            raise ImportError(
                "IndicTransToolkit or transformers not installed.\n"
                "pip install git+https://github.com/AI4Bharat/IndicTransToolkit.git\n"
                "pip install transformers torch sentencepiece"
            ) from exc

    # ── Translate ────────────────────────────────────────────────────────────

    def translate(
        self,
        text: str,
        source_lang: str,
        target_lang: str,
    ) -> str:
        """
        Translate text between supported languages via IndicTrans2.

        Args:
            text:        Input text.
            source_lang: ISO code ('en', 'ta', 'hi').
            target_lang: ISO code ('en', 'ta', 'hi').

        Returns:
            Translated text string.
        """
        if source_lang == target_lang or not text.strip():
            return text

        src_flores = self._FLORES_CODES[source_lang]
        tgt_flores = self._FLORES_CODES[target_lang]
        direction  = "indic-en" if target_lang == "en" else "en-indic"

        # Lazy load the appropriate pipeline
        if direction == "indic-en" and self._indic2en_pipeline is None:
            self._indic2en_pipeline = self._load_pipeline("indic-en")
        if direction == "en-indic" and self._en2indic_pipeline is None:
            self._en2indic_pipeline = self._load_pipeline("en-indic")

        pipeline = (
            self._indic2en_pipeline
            if direction == "indic-en"
            else self._en2indic_pipeline
        )
        tokenizer, model, device = pipeline

        import torch

        # Chunk large texts to avoid OOM
        chunks    = _split_into_chunks(text, config.TRANSLATION_MAX_CHARS)
        translated_chunks: list[str] = []

        for chunk in chunks:
            inputs = tokenizer(
                [chunk],
                src_lang=src_flores,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512,
            ).to(device)

            with torch.no_grad():
                output_tokens = model.generate(
                    **inputs,
                    forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_flores),
                    max_new_tokens=512,
                )
            decoded = tokenizer.batch_decode(
                output_tokens, skip_special_tokens=True
            )
            translated_chunks.append(decoded[0])

        return " ".join(translated_chunks)


# ─────────────────────────────────────────────
# TRANSLATION ENGINE — Google Translate Fallback
# ─────────────────────────────────────────────

class GoogleTranslateEngine:
    """
    Lightweight fallback translation engine using deep-translator (googletrans).
    No API key required for moderate usage.
    """

    _LANG_MAP = {"en": "en", "ta": "ta", "hi": "hi"}

    def translate(
        self,
        text: str,
        source_lang: str,
        target_lang: str,
    ) -> str:
        """
        Translate text using Google Translate via deep-translator.

        Args:
            text:        Input text.
            source_lang: ISO code.
            target_lang: ISO code.

        Returns:
            Translated text.
        """
        if source_lang == target_lang or not text.strip():
            return text

        try:
            from deep_translator import GoogleTranslator
        except ImportError as exc:
            raise ImportError(
                "deep-translator not installed.\npip install deep-translator"
            ) from exc

        chunks = _split_into_chunks(text, 1000)  # Reduced chunk size to prevent API errors
        results: list[str] = []
        for chunk in chunks:
            translated = None
            for attempt in range(3):
                try:
                    translated = GoogleTranslator(
                        source=self._LANG_MAP[source_lang],
                        target=self._LANG_MAP[target_lang],
                    ).translate(chunk)
                    
                    import time
                    time.sleep(1) # Prevent getting banned by free API
                    break
                except Exception as exc:
                    if attempt < 2:
                        import time
                        logger.warning("Google Translate API rate limited. Retrying in 3 seconds... (%s)", exc)
                        time.sleep(3)
                    else:
                        logger.error("Google Translate API completely failed: %s", exc)
                        raise ValueError("Translation failed due to API limits. Try a shorter document or run IndicTrans2.") from exc
            results.append(translated or chunk)
        return " ".join(results)


# ─────────────────────────────────────────────
# TRANSLATOR — PUBLIC INTERFACE
# ─────────────────────────────────────────────

class Translator:
    """
    Public façade for language detection and translation.

    Chooses the backend specified in config.TRANSLATION_BACKEND and
    falls back to GoogleTranslate if IndicTrans2 is unavailable.
    """

    def __init__(self) -> None:
        self._engine = self._build_engine()

    def _build_engine(self):
        """Instantiate the configured translation engine."""
        if config.TRANSLATION_BACKEND == "indictrans2":
            try:
                engine = IndicTrans2Engine()
                logger.info("Translation backend: IndicTrans2")
                return engine
            except Exception as exc:
                logger.warning(
                    "IndicTrans2 unavailable (%s). Falling back to Google Translate.",
                    exc,
                )
        logger.info("Translation backend: Google Translate (deep-translator)")
        return GoogleTranslateEngine()

    # ── Public methods ───────────────────────────────────────────────────────

    def detect_language(self, text: str) -> str:
        """
        Detect the language of the input text.

        Args:
            text: Input string.

        Returns:
            ISO 639-1 code ('en', 'ta', 'hi').
        """
        return detect_language(text)

    def to_english(self, text: str, source_lang: str) -> str:
        """
        Translate text to English from the given source language.
        No-op if source_lang is already 'en'.

        Args:
            text:        Input text.
            source_lang: Source language ISO code.

        Returns:
            English text.
        """
        if source_lang == "en":
            return text
        logger.info(
            "Translating %s → English (%d chars).",
            config.SUPPORTED_LANGUAGES.get(source_lang, source_lang),
            len(text),
        )
        with Timer(f"Translation {source_lang}→en", logger):
            return self._engine.translate(text, source_lang, "en")

    def from_english(self, text: str, target_lang: str) -> str:
        """
        Translate text from English to the given target language.
        No-op if target_lang is 'en'.

        Args:
            text:        English text.
            target_lang: Target language ISO code.

        Returns:
            Text in the target language.
        """
        if target_lang == "en":
            return text
        logger.info(
            "Translating English → %s (%d chars).",
            config.SUPPORTED_LANGUAGES.get(target_lang, target_lang),
            len(text),
        )
        with Timer(f"Translation en→{target_lang}", logger):
            return self._engine.translate(text, "en", target_lang)


# ─────────────────────────────────────────────
# PRIVATE HELPERS
# ─────────────────────────────────────────────

def _split_into_chunks(text: str, max_chars: int) -> list[str]:
    """
    Split text into chunks of at most `max_chars` characters, splitting
    preferentially at sentence boundaries.

    Args:
        text:      Input text.
        max_chars: Maximum characters per chunk.

    Returns:
        List of text chunks.
    """
    if len(text) <= max_chars:
        return [text]

    # Split on sentence-ending punctuation
    sentences = re.split(r"(?<=[.!?।॥\n])\s+", text)
    chunks:   list[str] = []
    current = ""

    for sentence in sentences:
        if len(current) + len(sentence) + 1 <= max_chars:
            current = (current + " " + sentence).strip()
        else:
            if current:
                chunks.append(current)
            # If a single sentence exceeds max_chars, hard-split it
            while len(sentence) > max_chars:
                chunks.append(sentence[:max_chars])
                sentence = sentence[max_chars:]
            current = sentence

    if current:
        chunks.append(current)

    return chunks


### `preprocessor.py`

In [ ]:
%%writefile preprocessor.py
"""
preprocessor.py
---------------
Text cleaning and preprocessing module.

Responsibilities:
  1. Remove page headers, footers, and page-number lines.
  2. Remove duplicate whitespace, broken OCR artefacts, and control chars.
  3. Normalise Unicode to NFC form.
  4. PRESERVE all legally-meaningful content:
       - Section / Clause / Article identifiers
       - GO numbers (G.O. Ms. No. …)
       - Act names, Rule citations
       - Dates and notification numbers
       - Paragraph structure

Design principle: When in doubt, keep the text. Legal documents must never
have meaningful content silently removed.
"""

import re
import unicodedata
import logging
from dataclasses import dataclass

import config
from utils import setup_logging, normalize_unicode, collapse_whitespace

logger = setup_logging(__name__)


# ─────────────────────────────────────────────
# REGEX PATTERNS — NOISE TO REMOVE
# ─────────────────────────────────────────────

# Standalone page numbers (e.g., "- 3 -", "Page 3", "3")
_RE_PAGE_NUMBER = re.compile(
    r"(?m)^[\-–—]?\s*Page\s*\d+\s*[\-–—]?$"
    r"|^[\-–—]\s*\d+\s*[\-–—]$"
    r"|^\d+$",
    re.IGNORECASE,
)

# Common header/footer watermarks and artefacts
_RE_HEADER_FOOTER = re.compile(
    r"(?m)^(CONFIDENTIAL|DRAFT|RESTRICTED|OFFICIAL USE ONLY|NOT FOR DISTRIBUTION)$",
    re.IGNORECASE,
)

# Excessive dashes / underscores used as visual separators (≥ 5 repetitions)
_RE_VISUAL_SEPARATOR = re.compile(r"[-_=*~]{5,}")

# Repeated dots or ellipsis artefacts (OCR artefact)
_RE_REPEATED_DOTS = re.compile(r"\.{4,}")

# Garbled single-char OCR lines (isolated characters that are not meaningful)
_RE_GARBLED_LINE = re.compile(r"(?m)^[^\w\n]{1,3}$")

# Null bytes and other non-printable control characters (except \n \r \t)
_RE_CONTROL_CHARS = re.compile(r"[^\S\n\r\t\x20]|\x00")

# Mixed script OCR artifacts: sequences of 3+ special symbols with no letters
_RE_OCR_NOISE = re.compile(r"[^a-zA-Z\u0B80-\u0BFF\u0900-\u097F\d\s\.\,\;\:\!\?\-\(\)\[\]\"\'\/\\]{3,}")

# ─────────────────────────────────────────────
# PATTERNS TO PRESERVE (whitelist guard)
# ─────────────────────────────────────────────

# These patterns mark lines that MUST be kept regardless of noise heuristics.
_PRESERVE_PATTERNS = [
    re.compile(r"G\.O\.|Government Order|G\.O\s*No", re.IGNORECASE),
    re.compile(r"\bSection\b|\bClause\b|\bArticle\b|\bRule\b|\bChapter\b", re.IGNORECASE),
    re.compile(r"\b(?:Act|Circular|Notification|Order|Gazette)\b", re.IGNORECASE),
    re.compile(r"\b\d{1,2}[\/\-]\d{1,2}[\/\-]\d{2,4}\b"),  # Dates
    re.compile(r"\bNo\.\s*\d+"),
    re.compile(r"\bRs\.?\s*\d+"),
    re.compile(r"\bProclaimed|Enacted|Whereas|In exercise"),
]


def _is_protected_line(line: str) -> bool:
    """Return True if a line matches any preservation pattern."""
    return any(p.search(line) for p in _PRESERVE_PATTERNS)


# ─────────────────────────────────────────────
# DATA MODEL
# ─────────────────────────────────────────────

@dataclass
class PreprocessResult:
    """
    Result of the preprocessing step.

    Attributes:
        cleaned_text:      Cleaned, normalised document text.
        original_length:   Character count of the raw text before cleaning.
        cleaned_length:    Character count after cleaning.
        removed_chars:     Characters removed (original - cleaned).
        go_numbers_found:  List of GO numbers identified in the document.
    """
    cleaned_text:    str
    original_length: int
    cleaned_length:  int
    removed_chars:   int
    go_numbers_found: list[str]

    @property
    def reduction_percent(self) -> float:
        """Percentage of characters removed during cleaning."""
        if self.original_length == 0:
            return 0.0
        return round((self.removed_chars / self.original_length) * 100, 2)


# ─────────────────────────────────────────────
# PREPROCESSOR CLASS
# ─────────────────────────────────────────────

class TextPreprocessor:
    """
    Pipeline of text-cleaning transformations applied sequentially.

    Each private method is a single-responsibility transform.
    They are applied in the order listed in `clean()`.
    """

    def clean(self, raw_text: str) -> PreprocessResult:
        """
        Run all preprocessing steps on the raw extracted text.

        Args:
            raw_text: Raw text from OCR or direct extraction.

        Returns:
            PreprocessResult with cleaned text and statistics.
        """
        original_length = len(raw_text)
        logger.info(
            "Starting preprocessing. Input: %d chars.", original_length
        )

        text = raw_text

        # Step 1 — Normalise Unicode
        text = self._normalize_unicode(text)

        # Step 2 — Remove control characters
        text = self._remove_control_chars(text)

        # Step 3 — Remove page numbers (line-by-line, protect preserved lines)
        text = self._remove_page_numbers(text)

        # Step 4 — Remove header/footer watermarks
        text = self._remove_headers_footers(text)

        # Step 5 — Remove visual separators (----, ====, etc.)
        text = self._remove_visual_separators(text)

        # Step 6 — Remove repeated dot artefacts
        text = self._remove_repeated_dots(text)

        # Step 7 — Remove garbled OCR lines
        text = self._remove_garbled_lines(text)

        # Step 8 — Collapse whitespace
        text = self._collapse_whitespace(text)

        # Step 9 — Final Unicode normalisation
        text = normalize_unicode(text).strip()

        cleaned_length = len(text)
        removed        = original_length - cleaned_length

        # Extract GO numbers for metadata
        go_numbers = self._extract_go_numbers(text)

        logger.info(
            "Preprocessing complete. Cleaned: %d chars (-%d, %.1f%% reduction). "
            "GO numbers found: %s",
            cleaned_length,
            removed,
            (removed / original_length * 100) if original_length else 0,
            go_numbers or "none",
        )

        return PreprocessResult(
            cleaned_text=text,
            original_length=original_length,
            cleaned_length=cleaned_length,
            removed_chars=max(0, removed),
            go_numbers_found=go_numbers,
        )

    # ── Transform methods ────────────────────────────────────────────────────

    @staticmethod
    def _normalize_unicode(text: str) -> str:
        """NFC-normalise and standardise curly quotes / dashes."""
        text = unicodedata.normalize("NFC", text)
        # Standardise typographic quotes to ASCII
        text = text.replace("\u2018", "'").replace("\u2019", "'")
        text = text.replace("\u201c", '"').replace("\u201d", '"')
        # Standardise em-dash / en-dash to hyphen-minus
        text = text.replace("\u2014", " - ").replace("\u2013", "-")
        # Non-breaking spaces → regular spaces
        text = text.replace("\u00a0", " ").replace("\u202f", " ")
        return text

    @staticmethod
    def _remove_control_chars(text: str) -> str:
        """Remove null bytes and invisible control characters."""
        return _RE_CONTROL_CHARS.sub("", text)

    @staticmethod
    def _remove_page_numbers(text: str) -> str:
        """
        Remove standalone page-number lines while protecting legal content.
        Processes line-by-line to preserve paragraph structure.
        """
        lines = text.split("\n")
        cleaned: list[str] = []
        for line in lines:
            if _is_protected_line(line):
                cleaned.append(line)
            elif _RE_PAGE_NUMBER.match(line.strip()):
                logger.debug("Removed page number line: %r", line.strip())
                # Replace with blank line to preserve paragraph boundary
                cleaned.append("")
            else:
                cleaned.append(line)
        return "\n".join(cleaned)

    @staticmethod
    def _remove_headers_footers(text: str) -> str:
        """Remove common header/footer watermark strings."""
        lines = text.split("\n")
        cleaned: list[str] = []
        for line in lines:
            if _is_protected_line(line):
                cleaned.append(line)
            elif _RE_HEADER_FOOTER.match(line.strip()):
                logger.debug("Removed header/footer line: %r", line.strip())
                cleaned.append("")
            else:
                cleaned.append(line)
        return "\n".join(cleaned)

    @staticmethod
    def _remove_visual_separators(text: str) -> str:
        """Replace long dash/underscore separators with a single newline."""
        return _RE_VISUAL_SEPARATOR.sub("\n", text)

    @staticmethod
    def _remove_repeated_dots(text: str) -> str:
        """Collapse chains of 4+ dots to a single ellipsis."""
        return _RE_REPEATED_DOTS.sub("...", text)

    @staticmethod
    def _remove_garbled_lines(text: str) -> str:
        """
        Remove lines that consist entirely of non-word characters (OCR noise),
        but only if they are not protected.
        """
        lines = text.split("\n")
        cleaned: list[str] = []
        for line in lines:
            if _is_protected_line(line):
                cleaned.append(line)
            elif _RE_GARBLED_LINE.match(line.strip()):
                logger.debug("Removed garbled line: %r", line.strip())
                cleaned.append("")
            else:
                cleaned.append(line)
        return "\n".join(cleaned)

    @staticmethod
    def _collapse_whitespace(text: str) -> str:
        """Collapse multiple spaces/tabs; collapse 3+ newlines to 2."""
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        # Strip trailing spaces from each line
        lines = [line.rstrip() for line in text.split("\n")]
        return "\n".join(lines)

    @staticmethod
    def _extract_go_numbers(text: str) -> list[str]:
        """Extract GO numbers from cleaned text for metadata."""
        patterns = [
            r"G\.O\.\s*(?:Ms\.|Rt\.|P\.|No\.?)?\s*\d+",
            r"GO\s*(?:Ms\.|Rt\.|No\.?)?\s*\d+",
            r"Government\s+Order\s+(?:No\.?)?\s*\d+",
        ]
        found: set[str] = set()
        for pat in patterns:
            for match in re.finditer(pat, text, re.IGNORECASE):
                found.add(match.group().strip())
        return sorted(found)


### `chunker.py`

In [ ]:
%%writefile chunker.py
"""
chunker.py
----------
Structure-aware document chunking module.

Responsibilities:
  1. Parse the cleaned document text and identify structural boundaries
     (Chapter, Section, Clause, GO Number, Article, Rule, etc.).
  2. Split text into semantically coherent chunks that align with
     the document's own structure rather than arbitrary character limits.
  3. Attach rich metadata to every chunk for citation purposes.
  4. Ensure no chunk exceeds CHUNK_SOFT_MAX_TOKENS (soft limit; structure wins).
  5. Apply token-level overlap between adjacent chunks for context continuity.

Design principle: A single legal clause must never be split across chunks.
Structure always takes priority over token-count targets.
"""

import re
import logging
from dataclasses import dataclass, field
from typing import Optional

import config
from utils import setup_logging, count_tokens_approx

logger = setup_logging(__name__)


# ─────────────────────────────────────────────
# DATA MODEL
# ─────────────────────────────────────────────

@dataclass
class DocumentChunk:
    """
    A single semantically coherent chunk of a legal document.

    Attributes:
        chunk_id:       Unique integer index across all chunks of a document.
        text:           The text content of this chunk.
        section_label:  Detected heading (e.g., "Section 4", "CHAPTER II").
        page_numbers:   Page(s) from which this chunk originated.
        token_count:    Approximate token count.
        char_start:     Character offset in the full document text (start).
        char_end:       Character offset in the full document text (end).
        metadata:       Arbitrary key-value metadata (GO numbers, dates, etc.)
    """
    chunk_id:     int
    text:         str
    section_label: Optional[str] = None
    page_numbers: list[int] = field(default_factory=list)
    token_count:  int = 0
    char_start:   int = 0
    char_end:     int = 0
    metadata:     dict = field(default_factory=dict)

    def __post_init__(self) -> None:
        if not self.token_count:
            self.token_count = count_tokens_approx(self.text)

    @property
    def short_citation(self) -> str:
        """
        Human-readable citation string.
        Example: 'Section 4.2 | Pages: 8-9'
        """
        parts = []
        if self.section_label:
            parts.append(self.section_label)
        if self.page_numbers:
            if len(self.page_numbers) == 1:
                parts.append(f"Page {self.page_numbers[0]}")
            else:
                parts.append(
                    f"Pages {self.page_numbers[0]}-{self.page_numbers[-1]}"
                )
        return " | ".join(parts) if parts else f"Chunk {self.chunk_id}"


# ─────────────────────────────────────────────
# COMPILED STRUCTURAL PATTERNS
# ─────────────────────────────────────────────

# Each pattern: (compiled_regex, label_format_string)
# The first capture group (\1) is used as the section label.
_STRUCTURAL_PATTERNS: list[tuple[re.Pattern, str]] = [
    # GO / G.O. numbers
    (re.compile(r"(?m)^(G\.O\.\s*(?:Ms\.|Rt\.|P\.|No\.?)?\s*\d+[^\n]*)"),     "GO"),
    (re.compile(r"(?m)^(GO\s*(?:Ms\.|Rt\.|No\.?)?\s*\d+[^\n]*)"),             "GO"),
    # Chapters
    (re.compile(r"(?m)^(CHAPTER\s+[IVXLCDM0-9]+\b[^\n]*)"),                    "CHAPTER"),
    (re.compile(r"(?m)^(Chapter\s+[IVXLCDM0-9]+\b[^\n]*)"),                    "Chapter"),
    # Parts
    (re.compile(r"(?m)^(PART\s+[IVXLCDM0-9]+\b[^\n]*)"),                       "PART"),
    # Sections (numeric, possibly hierarchical: 4, 4.1, 4.1.2)
    (re.compile(r"(?m)^(SECTION\s+\d+(?:\.\d+)*\b[^\n]*)"),                    "SECTION"),
    (re.compile(r"(?m)^(Section\s+\d+(?:\.\d+)*\b[^\n]*)"),                    "Section"),
    # Articles
    (re.compile(r"(?m)^(ARTICLE\s+\d+(?:\.\d+)*\b[^\n]*)"),                    "ARTICLE"),
    (re.compile(r"(?m)^(Article\s+\d+(?:\.\d+)*\b[^\n]*)"),                    "Article"),
    # Clauses
    (re.compile(r"(?m)^(CLAUSE\s+\d+(?:\.\d+)*\b[^\n]*)"),                     "CLAUSE"),
    (re.compile(r"(?m)^(Clause\s+\d+(?:\.\d+)*\b[^\n]*)"),                     "Clause"),
    # Rules
    (re.compile(r"(?m)^(RULE\s+\d+(?:\.\d+)*\b[^\n]*)"),                       "RULE"),
    (re.compile(r"(?m)^(Rule\s+\d+(?:\.\d+)*\b[^\n]*)"),                       "Rule"),
    # Numbered top-level items: "1. Title" or "12. Title"
    (re.compile(r"(?m)^(\d{1,3}\.\s+[A-Z][^\n]{5,})"),                         "Item"),
    # Sub-items in parentheses: "(1) Text"
    (re.compile(r"(?m)^(\(\d+\)\s+[A-Z][^\n]{5,})"),                           "Sub-item"),
    # Notification numbers
    (re.compile(r"(?m)^(Notification\s+No\.\s*\d+[^\n]*)"),                    "Notification"),
    (re.compile(r"(?m)^(NOTIFICATION\s+No\.\s*\d+[^\n]*)"),                    "Notification"),
    # Circular numbers
    (re.compile(r"(?m)^(Circular\s+No\.\s*\d+[^\n]*)"),                        "Circular"),
    # Schedule markers
    (re.compile(r"(?m)^(SCHEDULE\s+[IVXLCDM0-9]*\b[^\n]*)"),                   "Schedule"),
    (re.compile(r"(?m)^(THE\s+[A-Z ]{4,}\s+ACT[,\s]\d{4}[^\n]*)"),            "Act"),
]


def _find_splits(text: str) -> list[tuple[int, str]]:
    """
    Find all positions in `text` where a structural boundary starts.

    Args:
        text: Full document text.

    Returns:
        Sorted list of (char_position, label) tuples.
    """
    splits: dict[int, str] = {}
    for pattern, label in _STRUCTURAL_PATTERNS:
        for m in pattern.finditer(text):
            pos = m.start()
            if pos not in splits:
                splits[pos] = m.group(1).strip()  # First match wins
    return sorted(splits.items())


# ─────────────────────────────────────────────
# PAGE MAP BUILDER
# ─────────────────────────────────────────────

def _build_char_to_page_map(pages: list) -> list[tuple[int, int]]:
    """
    Build a list mapping character positions to page numbers.

    Args:
        pages: List of PageResult objects (must have .text and .page_number).

    Returns:
        List of (cumulative_char_end, page_number) tuples, sorted ascending.
    """
    mapping: list[tuple[int, int]] = []
    pos = 0
    for page in pages:
        text_len = len(page.text) + 2  # +2 for "\n\n" separator
        pos += text_len
        mapping.append((pos, page.page_number))
    return mapping


def _chars_to_pages(
    start: int,
    end: int,
    char_page_map: list[tuple[int, int]],
) -> list[int]:
    """
    Return the page number(s) that overlap with [start, end] character range.

    Args:
        start:         Character start of a chunk.
        end:           Character end of a chunk.
        char_page_map: Output of _build_char_to_page_map.

    Returns:
        Sorted list of unique page numbers.
    """
    pages: set[int] = set()
    prev_boundary = 0
    for boundary, page_num in char_page_map:
        if start < boundary and end > prev_boundary:
            pages.add(page_num)
        prev_boundary = boundary
    return sorted(pages)


# ─────────────────────────────────────────────
# CHUNKER CLASS
# ─────────────────────────────────────────────

class StructureAwareChunker:
    """
    Splits a document into semantically coherent chunks using structural
    anchors (Section, Chapter, GO Number, etc.) rather than fixed sizes.

    Algorithm:
        1. Scan text for all structural split points.
        2. Slice text at each split point to create candidate sections.
        3. If a section exceeds CHUNK_SOFT_MAX_TOKENS, sub-split at paragraph
           boundaries (double newlines).
        4. Add token-level overlap with the previous chunk for context.
        5. Merge sections smaller than CHUNK_SOFT_MIN_TOKENS with the next one.
    """

    def __init__(self) -> None:
        self._max_tokens  = config.CHUNK_SOFT_MAX_TOKENS
        self._min_tokens  = config.CHUNK_SOFT_MIN_TOKENS
        self._overlap     = config.CHUNK_OVERLAP_TOKENS

    def chunk(
        self,
        full_text: str,
        pages: Optional[list] = None,
    ) -> list[DocumentChunk]:
        """
        Chunk the document text.

        Args:
            full_text: Cleaned full document text.
            pages:     Optional list of PageResult objects for page mapping.

        Returns:
            List of DocumentChunk objects in document order.
        """
        if not full_text.strip():
            logger.warning("Chunker received empty text.")
            return []

        logger.info(
            "Chunking document: %d chars, ~%d tokens.",
            len(full_text),
            count_tokens_approx(full_text),
        )

        char_page_map = _build_char_to_page_map(pages) if pages else []

        # Step 1 — Find structural boundaries
        splits = _find_splits(full_text)

        # Step 2 — Slice into raw sections
        raw_sections = self._slice_into_sections(full_text, splits)

        # Step 3 — Sub-split oversized sections and apply overlap
        chunks: list[DocumentChunk] = []
        chunk_id = 0
        prev_overlap_text = ""

        for section_text, section_label, char_start in raw_sections:
            sub_blocks = self._sub_split(section_text)

            for i, block in enumerate(sub_blocks):
                # Prepend overlap from previous block
                if prev_overlap_text and i == 0:
                    combined = prev_overlap_text.strip() + "\n\n" + block.strip()
                else:
                    combined = block.strip()

                if not combined.strip():
                    continue

                # Merge tiny fragments with previous chunk
                if (
                    count_tokens_approx(combined) < self._min_tokens
                    and chunks
                    and i == 0
                ):
                    chunks[-1].text += "\n\n" + combined
                    chunks[-1].token_count = count_tokens_approx(chunks[-1].text)
                    continue

                # Compute character range
                block_start = char_start + section_text.find(block)
                block_end   = block_start + len(block)

                page_numbers = _chars_to_pages(
                    block_start, block_end, char_page_map
                ) if char_page_map else []

                chunk = DocumentChunk(
                    chunk_id      = chunk_id,
                    text          = combined,
                    section_label = section_label.strip() if section_label else None,
                    page_numbers  = page_numbers,
                    char_start    = block_start,
                    char_end      = block_end,
                )
                chunks.append(chunk)
                chunk_id += 1

                # Compute overlap for next iteration
                words = combined.split()
                overlap_words = words[-self._overlap:] if len(words) > self._overlap else words
                prev_overlap_text = " ".join(overlap_words)

        # Step 4 — Final merge: merge consecutive tiny chunks
        chunks = self._merge_small_chunks(chunks)

        logger.info(
            "Chunking complete. Produced %d chunk(s).", len(chunks)
        )
        for c in chunks:
            logger.debug(
                "  Chunk %d | label=%r | tokens=%d | pages=%s",
                c.chunk_id, c.section_label, c.token_count, c.page_numbers,
            )

        return chunks

    # ── Private helpers ──────────────────────────────────────────────────────

    def _slice_into_sections(
        self,
        text: str,
        splits: list[tuple[int, str]],
    ) -> list[tuple[str, Optional[str], int]]:
        """
        Slice text at structural split positions.

        Returns:
            List of (section_text, label, char_start) tuples.
        """
        if not splits:
            # No structural boundaries found — treat entire doc as one section
            return [(text, None, 0)]

        sections: list[tuple[str, Optional[str], int]] = []
        boundaries = [(0, None)] + [(pos, lbl) for pos, lbl in splits]

        for idx in range(len(boundaries)):
            start_pos, label = boundaries[idx]
            end_pos = (
                boundaries[idx + 1][0] if idx + 1 < len(boundaries) else len(text)
            )
            section_text = text[start_pos:end_pos]
            if section_text.strip():
                sections.append((section_text, label, start_pos))

        return sections

    def _sub_split(self, text: str) -> list[str]:
        """
        If `text` exceeds CHUNK_SOFT_MAX_TOKENS, sub-split at paragraph
        boundaries (blank lines). Falls back to sentence boundaries.

        Args:
            text: Section text.

        Returns:
            List of sub-blocks, each within the soft max tokens.
        """
        if count_tokens_approx(text) <= self._max_tokens:
            return [text]

        # Try paragraph splitting first
        paragraphs = re.split(r"\n{2,}", text)
        blocks: list[str] = []
        current = ""

        for para in paragraphs:
            candidate = (current + "\n\n" + para).strip() if current else para
            if count_tokens_approx(candidate) <= self._max_tokens:
                current = candidate
            else:
                if current:
                    blocks.append(current)
                # If the paragraph itself is too large, split by sentences
                if count_tokens_approx(para) > self._max_tokens:
                    blocks.extend(self._split_by_sentences(para))
                    current = ""
                else:
                    current = para

        if current:
            blocks.append(current)

        return blocks if blocks else [text]

    def _split_by_sentences(self, text: str) -> list[str]:
        """
        Emergency fallback: split by sentence-ending punctuation.

        Args:
            text: Paragraph text.

        Returns:
            List of sentence groups, each within soft max tokens.
        """
        sentences = re.split(r"(?<=[.!?।॥])\s+", text)
        groups: list[str] = []
        current = ""

        for sent in sentences:
            candidate = (current + " " + sent).strip() if current else sent
            if count_tokens_approx(candidate) <= self._max_tokens:
                current = candidate
            else:
                if current:
                    groups.append(current)
                current = sent

        if current:
            groups.append(current)

        return groups if groups else [text]

    def _merge_small_chunks(
        self,
        chunks: list[DocumentChunk],
    ) -> list[DocumentChunk]:
        """
        Merge consecutive chunks that are below the minimum token threshold.

        Args:
            chunks: List of DocumentChunk objects.

        Returns:
            New list with small chunks merged.
        """
        if not chunks:
            return chunks

        merged: list[DocumentChunk] = [chunks[0]]
        for chunk in chunks[1:]:
            if (
                merged[-1].token_count < self._min_tokens
                and not merged[-1].section_label
            ):
                merged[-1].text       += "\n\n" + chunk.text
                merged[-1].token_count = count_tokens_approx(merged[-1].text)
                merged[-1].char_end    = chunk.char_end
                if chunk.page_numbers:
                    for p in chunk.page_numbers:
                        if p not in merged[-1].page_numbers:
                            merged[-1].page_numbers.append(p)
            else:
                merged.append(chunk)

        return merged


### `embeddings.py`

In [ ]:
%%writefile embeddings.py
"""
embeddings.py
-------------
Embedding generation module.

Responsibilities:
  1. Load a multilingual dense retrieval model (BAAI/bge-m3 or multilingual-e5-large).
  2. Generate embeddings for a list of text strings in batches.
  3. Optionally L2-normalise embeddings for cosine similarity via dot product.
  4. Provide helper to embed a single query string.

Design decisions:
  - sentence-transformers is used because it abstracts model loading, batching,
    and normalisation uniformly across HuggingFace models.
  - The model is loaded lazily on first use to keep startup time fast.
  - BAAI/bge-m3 is preferred: it supports 100+ languages (including Tamil and Hindi)
    and produces state-of-the-art dense retrieval vectors at 1024 dimensions.
"""

import logging
import numpy as np
from typing import Optional

import config
from utils import setup_logging, Timer

logger = setup_logging(__name__)


# ─────────────────────────────────────────────
# EMBEDDING MODEL
# ─────────────────────────────────────────────

class EmbeddingModel:
    """
    Wrapper around sentence-transformers for document / query embedding.

    Usage:
        model = EmbeddingModel()
        doc_vecs  = model.embed_documents(["text1", "text2"])
        query_vec = model.embed_query("What is the retirement age?")
    """

    def __init__(self) -> None:
        self._model   = None
        self._model_name: Optional[str] = None
        logger.info(
            "EmbeddingModel created. Model '%s' will load on first use.",
            config.EMBEDDING_MODEL_NAME,
        )

    # ── Lazy loader ──────────────────────────────────────────────────────────

    def _ensure_loaded(self) -> None:
        """Load the SentenceTransformer model if not already loaded."""
        if self._model is not None:
            return

        for model_name in [
            config.EMBEDDING_MODEL_NAME,
            config.EMBEDDING_FALLBACK_MODEL,
        ]:
            try:
                from sentence_transformers import SentenceTransformer
                logger.info("Loading embedding model: %s …", model_name)
                with Timer(f"Load {model_name}", logger):
                    self._model = SentenceTransformer(
                        model_name,
                        device="cpu",
                        cache_folder=str(config.MODELS_DIR / "sentence_transformers"),
                    )
                self._model_name = model_name
                logger.info(
                    "Embedding model '%s' loaded successfully (dim=%d).",
                    model_name,
                    self._model.get_sentence_embedding_dimension(),
                )
                return
            except Exception as exc:
                logger.warning(
                    "Failed to load model '%s': %s. Trying next.", model_name, exc
                )

        raise RuntimeError(
            "Could not load any embedding model. "
            f"Tried: {config.EMBEDDING_MODEL_NAME}, {config.EMBEDDING_FALLBACK_MODEL}\n"
            "Install with: pip install sentence-transformers"
        )

    # ── Public API ────────────────────────────────────────────────────────────

    @property
    def dimension(self) -> int:
        """Return the embedding vector dimension."""
        self._ensure_loaded()
        return self._model.get_sentence_embedding_dimension()

    @property
    def model_name(self) -> str:
        """Return the active model name."""
        self._ensure_loaded()
        return self._model_name

    def embed_documents(self, texts: list[str]) -> np.ndarray:
        """
        Generate embeddings for a list of document texts.

        Args:
            texts: List of text strings to embed.

        Returns:
            np.ndarray of shape (N, D) where N=len(texts), D=embedding dim.
            Embeddings are L2-normalised if config.NORMALIZE_EMBEDDINGS is True.

        Raises:
            ValueError: If `texts` is empty.
            RuntimeError: If the embedding model cannot be loaded.
        """
        if not texts:
            raise ValueError("Cannot embed an empty list of texts.")

        self._ensure_loaded()
        logger.info(
            "Embedding %d document chunk(s) with '%s'.",
            len(texts),
            self._model_name,
        )

        with Timer(f"Embed {len(texts)} docs", logger):
            vectors = self._model.encode(
                texts,
                batch_size=config.EMBEDDING_BATCH_SIZE,
                show_progress_bar=len(texts) > 10,
                normalize_embeddings=config.NORMALIZE_EMBEDDINGS,
                convert_to_numpy=True,
            )

        logger.info(
            "Document embeddings generated: shape %s.", vectors.shape
        )
        return vectors.astype(np.float32)

    def embed_query(self, query: str) -> np.ndarray:
        """
        Generate an embedding for a single query string.

        BGE-M3 uses a special instruction prefix for queries in its asymmetric
        retrieval setup; this is handled automatically by sentence-transformers
        if the model's tokeniser includes the instruction.

        Args:
            query: The user's question (in English, post-translation).

        Returns:
            np.ndarray of shape (D,) — 1-D vector.
        """
        if not query.strip():
            raise ValueError("Query string is empty.")

        self._ensure_loaded()
        logger.debug("Embedding query: %r", query[:80])

        with Timer("Embed query", logger):
            vector = self._model.encode(
                query,
                normalize_embeddings=config.NORMALIZE_EMBEDDINGS,
                convert_to_numpy=True,
            )

        return vector.astype(np.float32)

    def embed_texts(self, texts: list[str]) -> np.ndarray:
        """
        Alias for embed_documents; accepts any list of texts.

        Args:
            texts: List of strings.

        Returns:
            np.ndarray of shape (N, D).
        """
        return self.embed_documents(texts)


### `retriever.py`

In [ ]:
%%writefile retriever.py
"""
retriever.py
------------
Vector-store and retrieval module using FAISS.

Responsibilities:
  1. Build an in-memory FAISS index from document chunk embeddings.
  2. Retrieve the top-K most relevant chunks for a given query embedding.
  3. Return ranked chunks with similarity scores.
  4. Never persist the index to disk — memory-only, per config constraints.

Design decisions:
  - FAISS IndexFlatIP (inner product) is used because embeddings are
    L2-normalised; dot product on unit vectors equals cosine similarity.
  - IVF index is offered for large document collections (>10k chunks).
  - The retriever stores chunk references in a parallel list to FAISS,
    enabling O(1) lookup after search.
"""

import logging
import numpy as np
from typing import Optional

import config
from chunker import DocumentChunk
from embeddings import EmbeddingModel
from utils import setup_logging, Timer

logger = setup_logging(__name__)


# ─────────────────────────────────────────────
# RETRIEVAL RESULT
# ─────────────────────────────────────────────

class RetrievalResult:
    """
    Holds one retrieved chunk and its similarity score.

    Attributes:
        chunk:           The DocumentChunk object.
        score:           Cosine similarity (0-1); higher is more relevant.
        rank:            1-based rank in the retrieval list.
    """

    def __init__(
        self,
        chunk: DocumentChunk,
        score: float,
        rank: int,
    ) -> None:
        self.chunk = chunk
        self.score = round(float(score), 4)
        self.rank  = rank

    @property
    def score_percent(self) -> float:
        """Return score as a percentage (0-100)."""
        return round(self.score * 100, 2)

    def __repr__(self) -> str:
        return (
            f"RetrievalResult(rank={self.rank}, score={self.score:.4f}, "
            f"label={self.chunk.section_label!r})"
        )


# ─────────────────────────────────────────────
# FAISS INDEX
# ─────────────────────────────────────────────

def _import_faiss():
    """Lazy import FAISS."""
    try:
        import faiss
        return faiss
    except ImportError as exc:
        raise ImportError(
            "faiss-cpu is required for vector retrieval.\n"
            "Install with: pip install faiss-cpu"
        ) from exc


def _build_faiss_index(vectors: np.ndarray, index_type: str):
    """
    Build and return a FAISS index from an array of embedding vectors.

    Args:
        vectors:    np.ndarray of shape (N, D), float32.
        index_type: 'flat_ip' | 'flat_l2' | 'ivf'

    Returns:
        FAISS index populated with vectors.
    """
    faiss = _import_faiss()
    n, dim = vectors.shape
    logger.info(
        "Building FAISS index. Type: %s, vectors: %d, dim: %d.",
        index_type, n, dim,
    )

    if index_type == "flat_ip":
        index = faiss.IndexFlatIP(dim)          # Inner product (cosine)
    elif index_type == "flat_l2":
        index = faiss.IndexFlatL2(dim)
    elif index_type == "ivf":
        quantizer = faiss.IndexFlatIP(dim)
        nlist     = max(1, min(n // 10, 100))   # Adaptive number of cells
        index     = faiss.IndexIVFFlat(quantizer, dim, nlist)
        index.train(vectors)
    else:
        raise ValueError(f"Unknown FAISS index type: {index_type!r}")

    index.add(vectors)
    logger.info("FAISS index built. Total vectors: %d.", index.ntotal)
    return index


# ─────────────────────────────────────────────
# RETRIEVER CLASS
# ─────────────────────────────────────────────

class VectorRetriever:
    """
    In-memory vector retriever backed by FAISS.

    Typical workflow:
        retriever = VectorRetriever(embedding_model)
        retriever.index_chunks(chunks)
        results = retriever.retrieve("What is the retirement age?")
    """

    def __init__(self, embedding_model: EmbeddingModel) -> None:
        self._model:  EmbeddingModel             = embedding_model
        self._index                              = None
        self._chunks: list[DocumentChunk]        = []
        self._vectors: Optional[np.ndarray]      = None
        logger.info("VectorRetriever initialised.")

    # ── Indexing ─────────────────────────────────────────────────────────────

    def index_chunks(self, chunks: list[DocumentChunk]) -> None:
        """
        Generate embeddings for all chunks and build the FAISS index.

        Args:
            chunks: List of DocumentChunk objects from the chunker.

        Raises:
            ValueError: If chunks list is empty.
        """
        if not chunks:
            raise ValueError("Cannot index an empty list of chunks.")

        logger.info("Indexing %d chunks into FAISS.", len(chunks))
        self._chunks = chunks

        texts = [c.text for c in chunks]

        with Timer("Index all chunks", logger):
            self._vectors = self._model.embed_documents(texts)
            self._index   = _build_faiss_index(
                self._vectors, config.FAISS_INDEX_TYPE
            )

        logger.info(
            "Indexing complete. %d vectors stored in memory.", len(chunks)
        )

    def is_indexed(self) -> bool:
        """Return True if the index has been built."""
        return self._index is not None and bool(self._chunks)

    # ── Retrieval ─────────────────────────────────────────────────────────────

    def retrieve(
        self,
        query: str,
        top_k: Optional[int] = None,
        min_score: Optional[float] = None,
    ) -> list[RetrievalResult]:
        """
        Retrieve the most relevant chunks for a query.

        Args:
            query:     User question in English (post-translation).
            top_k:     Number of chunks to return (default: config.RETRIEVAL_TOP_K).
            min_score: Minimum similarity threshold (default: config.RETRIEVAL_MIN_SCORE).

        Returns:
            List of RetrievalResult objects, ordered by descending similarity.
            Empty list if no chunks meet the threshold.

        Raises:
            RuntimeError: If index_chunks() has not been called yet.
        """
        if not self.is_indexed():
            raise RuntimeError(
                "VectorRetriever: call index_chunks() before retrieve()."
            )

        k         = top_k     or config.RETRIEVAL_TOP_K
        threshold = min_score if min_score is not None else config.RETRIEVAL_MIN_SCORE

        logger.info(
            "Retrieving top-%d chunks for query: %r (threshold=%.2f)",
            k, query[:80], threshold,
        )

        with Timer("FAISS search", logger):
            query_vec = self._model.embed_query(query)
            query_vec = query_vec.reshape(1, -1)   # (1, D)
            scores, indices = self._index.search(query_vec, k)

        scores  = scores[0]    # Shape (k,)
        indices = indices[0]   # Shape (k,)

        results: list[RetrievalResult] = []
        for rank, (idx, score) in enumerate(zip(indices, scores), start=1):
            if idx == -1:        # FAISS pads with -1 for empty slots
                continue
            if float(score) < threshold:
                logger.debug(
                    "Chunk %d skipped (score %.4f < threshold %.4f).",
                    idx, score, threshold,
                )
                continue
            result = RetrievalResult(
                chunk=self._chunks[int(idx)],
                score=float(score),
                rank=rank,
            )
            results.append(result)
            logger.debug(
                "  Rank %d: chunk_id=%d, score=%.4f, label=%r",
                rank, int(idx), score, self._chunks[int(idx)].section_label,
            )

        logger.info(
            "Retrieved %d relevant chunk(s) (of %d candidates).",
            len(results), k,
        )
        return results

    # ── Utilities ─────────────────────────────────────────────────────────────

    def get_chunk_by_id(self, chunk_id: int) -> Optional[DocumentChunk]:
        """Return a chunk by its chunk_id, or None if not found."""
        for chunk in self._chunks:
            if chunk.chunk_id == chunk_id:
                return chunk
        return None

    @property
    def total_chunks(self) -> int:
        """Total number of indexed chunks."""
        return len(self._chunks)


### `rag_engine.py`

In [ ]:
%%writefile rag_engine.py
"""
rag_engine.py
-------------
Retrieval-Augmented Generation (RAG) engine.

Responsibilities:
  1. Accept a user question in English.
  2. Format a strict grounding prompt from retrieved chunks.
  3. Call the configured LLM reader (HuggingFace flan-t5 / OpenAI / Gemini).
  4. Return only the generated answer string — verification is handled in
     verification.py.

Design decisions:
  - Temperature is set to 0 to maximise factual consistency.
  - The system prompt explicitly forbids the model from using external knowledge.
  - Context is formatted with clear section labels so the model can cite them.
  - Multiple LLM backends are supported; HuggingFace Flan-T5 is the default
    because it is open-source and runs locally without API keys.
"""

import logging
from typing import Optional

import config
from retriever import RetrievalResult
from utils import setup_logging, Timer

logger = setup_logging(__name__)


# ─────────────────────────────────────────────
# CONTEXT BUILDER
# ─────────────────────────────────────────────

def build_context(results: list[RetrievalResult]) -> str:
    """
    Format a list of retrieval results into a numbered context block for the LLM.

    Each entry includes the section label and page numbers as a header so the
    model can reproduce them in its answer.

    Args:
        results: Ranked list of RetrievalResult objects.

    Returns:
        Multi-line context string.
    """
    context_parts: list[str] = []
    for r in results:
        header_parts = []
        if r.chunk.section_label:
            header_parts.append(r.chunk.section_label)
        if r.chunk.page_numbers:
            pages = ", ".join(str(p) for p in r.chunk.page_numbers)
            header_parts.append(f"Page(s): {pages}")
        header = " | ".join(header_parts) if header_parts else f"Chunk {r.chunk.chunk_id}"

        context_parts.append(f"[{r.rank}] {header}\n{r.chunk.text.strip()}")

    return "\n\n".join(context_parts)


# ─────────────────────────────────────────────
# LLM BACKENDS
# ─────────────────────────────────────────────

class HuggingFaceReader:
    """
    Reader using a HuggingFace seq2seq model (Flan-T5 by default).
    Runs entirely locally — no API key required.
    """

    _tokenizer = None
    _model = None

    @classmethod
    def _load_model(cls):
        if cls._model is None:
            try:
                from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
                import torch
                logger.info("Loading HuggingFace reader model: %s …", config.HF_READER_MODEL)
                cls._tokenizer = AutoTokenizer.from_pretrained(config.HF_READER_MODEL)
                cls._model = AutoModelForSeq2SeqLM.from_pretrained(config.HF_READER_MODEL)
                cls._model.eval()
                if torch.cuda.is_available():
                    cls._model = cls._model.to("cuda")
                logger.info("HuggingFace reader ready.")
            except ImportError as exc:
                raise ImportError(
                    "transformers and torch are required for HuggingFace backend."
                ) from exc

    def generate(self, prompt: str, system_prompt: str = "") -> str:
        """
        Generate an answer from a formatted prompt.
        """
        self._load_model()
        if system_prompt:
            prompt = f"{system_prompt}\n\n{prompt}"
        with Timer("HF generation", logger):
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
            inputs = self.__class__._tokenizer(prompt, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = self.__class__._model.generate(
                    **inputs,
                    max_new_tokens=config.LLM_MAX_NEW_TOKENS,
                    temperature=config.LLM_TEMPERATURE,
                    do_sample=False,
                    num_beams=3,
                    repetition_penalty=1.1,
                    no_repeat_ngram_size=5,
                    early_stopping=True,
                )
            answer = self.__class__._tokenizer.decode(outputs[0], skip_special_tokens=True)
        return answer.strip()


class OpenAIReader:
    """
    Reader using OpenAI Chat Completions API.
    Requires OPENAI_API_KEY in environment variables.
    """

    def generate(self, prompt: str, system_prompt: str = "") -> str:
        try:
            import os
            from openai import OpenAI

            api_key = os.environ.get("OPENAI_API_KEY", "")
            if not api_key:
                raise EnvironmentError(
                    "OPENAI_API_KEY environment variable not set."
                )
            client = OpenAI(api_key=api_key)
            with Timer("OpenAI generation", logger):
                response = client.chat.completions.create(
                    model=config.OPENAI_MODEL,
                    messages=[
                        {"role": "system", "content": system_prompt or config.RAG_SYSTEM_PROMPT},
                        {"role": "user",   "content": prompt},
                    ],
                    max_tokens=config.LLM_MAX_NEW_TOKENS,
                    temperature=config.LLM_TEMPERATURE,
                )
            return response.choices[0].message.content.strip()
        except ImportError as exc:
            raise ImportError(
                "openai package not installed.\npip install openai"
            ) from exc

class GeminiReader:
    """
    Reader using Google Gemini API (google-genai SDK).
    Requires GOOGLE_API_KEY in environment variables or .env file.
    """

    def generate(self, prompt: str, system_prompt: str = "") -> str:
        try:
            import os
            from google import genai
            from google.genai import types
            import httpx

            api_key = os.environ.get("GOOGLE_API_KEY", "")
            if not api_key:
                raise EnvironmentError(
                    "GOOGLE_API_KEY not set. Add it to the .env file:\n"
                    "  GOOGLE_API_KEY=your_key_here"
                )

            logger.info("Calling Gemini API (model=%s, timeout=30s)...", config.GEMINI_MODEL)

            # Use http_options for timeout (google-genai 2.x)
            client = genai.Client(api_key=api_key, http_options={'timeout': 30.0})
            full_prompt = f"{system_prompt or config.RAG_SYSTEM_PROMPT}\n\n{prompt}"

            with Timer("Gemini generation", logger):
                response = client.models.generate_content(
                    model=config.GEMINI_MODEL,
                    contents=full_prompt,
                    config=types.GenerateContentConfig(
                        temperature=config.LLM_TEMPERATURE,
                        max_output_tokens=config.LLM_MAX_NEW_TOKENS,
                    ),
                )
            return response.text.strip()

        except ImportError as exc:
            raise ImportError(
                "google-genai not installed.\npip install google-genai"
            ) from exc
        except Exception as exc:
            err = str(exc)
            if "API_KEY_INVALID" in err or "401" in err:
                raise RuntimeError(
                    "Gemini API key is INVALID. Please get a valid key from:\n"
                    "  https://aistudio.google.com/app/apikey\n"
                    "Then update your .env file: GOOGLE_API_KEY=AIzaSy..."
                ) from exc
            elif "timeout" in err.lower() or "timed out" in err.lower():
                raise RuntimeError(
                    "Gemini API timed out (30s). Check your internet connection."
                ) from exc
            else:
                raise RuntimeError(f"Gemini API error: {exc}") from exc


# ─────────────────────────────────────────────
# RAG ENGINE
# ─────────────────────────────────────────────

class RAGEngine:
    """
    Retrieval-Augmented Generation engine.

    Combines retrieved context with a strict grounding prompt and delegates
    to the configured LLM reader to produce an answer.

    Usage:
        engine = RAGEngine()
        answer = engine.generate(question_en, retrieval_results)
    """

    def __init__(self) -> None:
        self._reader = self._build_reader()

    def _build_reader(self):
        """Instantiate the LLM reader specified in config."""
        backend = config.LLM_BACKEND.lower()
        if backend == "openai":
            logger.info("RAGEngine using OpenAI reader.")
            return OpenAIReader()
        elif backend == "google":
            logger.info("RAGEngine using Gemini reader.")
            return GeminiReader()
        else:
            logger.info("RAGEngine using HuggingFace reader: %s", config.HF_READER_MODEL)
            return HuggingFaceReader()

    def generate(
        self,
        question_en: str,
        results: list[RetrievalResult],
        total_pages: int = 1,
    ) -> str:
        """
        Generate a grounded answer from retrieved chunks.

        Args:
            question_en: The user's question in English (post-translation).
            results:     Ranked RetrievalResult list from the retriever.

        Returns:
            Generated answer string (may be NOT_FOUND_RESPONSE if context
            is empty or model returns nothing useful).
        """
        if not results:
            logger.warning("RAGEngine received empty results — returning not-found.")
            return config.NOT_FOUND_RESPONSE

        context = build_context(results)
        prompt  = config.RAG_USER_PROMPT_TEMPLATE.format(
            context=context,
            question=question_en,
        )

        # Dynamically calculate min and max lines based on page count
        min_lines = max(7, 5 + (total_pages * 2))
        max_lines = max(10, min_lines + (total_pages * 3))
        system_prompt = config.RAG_SYSTEM_PROMPT.format(min_lines=min_lines, max_lines=max_lines)

        logger.debug("RAG prompt (first 300 chars): %s …", prompt[:300])

        try:
            with Timer("RAG full generation", logger):
                answer = self._reader.generate(prompt, system_prompt=system_prompt)
        except Exception as exc:
            logger.error("\n[!] LLM Generation failed: %s", exc)
            cprint(f"\n  [ERROR] {exc}", colour="red")
            return config.NOT_FOUND_RESPONSE

        if not answer or not answer.strip():
            logger.warning("LLM returned empty answer — returning not-found.")
            return config.NOT_FOUND_RESPONSE

        logger.info("Generated answer (%d chars): %s …", len(answer), answer[:80])
        return answer.strip()


### `verification.py`

In [ ]:
%%writefile verification.py
"""
verification.py
---------------
Answer verification and citation module.

This is the most critical module in the pipeline.

Responsibilities:
  1. Verify that EVERY factual claim in the generated answer is
     supported by the retrieved document chunks (NOT by model memory).
  2. Compute a confidence score for the answer.
  3. Return verified citation metadata: section, page, GO number.
  4. Reject the answer and return NOT_FOUND_RESPONSE if verification fails
     after one retry.

Verification algorithm (multi-stage):
  Stage 1 — Token Overlap Check:
      Compute the ratio of content words in the answer that also appear
      in the retrieved context. Below VERIFICATION_MIN_OVERLAP → FAIL.

  Stage 2 — Key Entity Check:
      Extract named entities from the answer (numbers, dates, GO refs,
      section refs). Verify each entity appears literally in the context.
      Any unsupported entity → FAIL.

  Stage 3 — Sentence Grounding Check:
      For each sentence in the answer, find the best-matching context
      sentence (ROUGE-L or token overlap). If any sentence has zero
      support → FAIL.

  Confidence score formula:
      (token_overlap_ratio × 40) +
      (entity_coverage   × 40) +
      (sentence_coverage × 20)

Design principle: It is far better to return "not found" than to return an
unverified answer in a legal/government document QA system.
"""

import re
import logging
import math
from dataclasses import dataclass, field
from typing import Optional

import config
from retriever import RetrievalResult
from utils import setup_logging, Timer

logger = setup_logging(__name__)


# ─────────────────────────────────────────────
# DATA MODELS
# ─────────────────────────────────────────────

@dataclass
class CitationInfo:
    """
    Citation metadata for a verified answer.

    Attributes:
        section_label:  Section/Clause label (e.g., "Section 4.2").
        page_numbers:   Page(s) in the source document.
        go_numbers:     GO numbers mentioned in the supporting context.
        chunk_ids:      Internal chunk IDs that support the answer.
        similarity_scores: Similarity scores of supporting chunks.
    """
    section_label:     Optional[str]  = None
    page_numbers:      list[int]       = field(default_factory=list)
    go_numbers:        list[str]       = field(default_factory=list)
    chunk_ids:         list[int]       = field(default_factory=list)
    similarity_scores: list[float]     = field(default_factory=list)

    def format(self) -> str:
        """Return a formatted citation string for display."""
        parts: list[str] = []
        if self.section_label:
            parts.append(f"Section: {self.section_label}")
        if self.page_numbers:
            parts.append("Page: " + ", ".join(str(p) for p in self.page_numbers))
        if self.go_numbers:
            parts.append("GO: " + ", ".join(self.go_numbers))
        return " | ".join(parts) if parts else "Document (section unknown)"


@dataclass
class VerificationResult:
    """
    Full result of the answer-verification step.

    Attributes:
        passed:          True if the answer is grounded and verified.
        answer:          The (possibly cleaned) verified answer, or NOT_FOUND.
        confidence:      0-100 confidence score.
        citation:        CitationInfo object.
        token_overlap:   Ratio of answer tokens found in context (0-1).
        entity_coverage: Ratio of extracted entities supported by context (0-1).
        sentence_coverage: Ratio of answer sentences with context support (0-1).
        failure_reason:  Human-readable reason for failure (if not passed).
    """
    passed:            bool
    answer:            str
    confidence:        float
    citation:          CitationInfo
    token_overlap:     float = 0.0
    entity_coverage:   float = 0.0
    sentence_coverage: float = 0.0
    failure_reason:    Optional[str] = None

    @property
    def status_label(self) -> str:
        return "PASSED ✓" if self.passed else "FAILED ✗"

    @property
    def confidence_str(self) -> str:
        return f"{self.confidence:.1f}%"


# ─────────────────────────────────────────────
# ENTITY EXTRACTION
# ─────────────────────────────────────────────

_ENTITY_PATTERNS = [
    # Numbers (cardinal, ordinal)
    re.compile(r"\b\d+(?:\.\d+)?\b"),
    # Dates: DD/MM/YYYY, DD-MM-YYYY
    re.compile(r"\b\d{1,2}[\/\-]\d{1,2}[\/\-]\d{2,4}\b"),
    # Year references
    re.compile(r"\b(?:19|20)\d{2}\b"),
    # GO references
    re.compile(r"G\.O\.\s*(?:Ms\.|Rt\.|No\.?)?\s*\d+", re.IGNORECASE),
    re.compile(r"GO\s*(?:No\.?)?\s*\d+", re.IGNORECASE),
    # Section/Clause/Article/Rule references in answer
    re.compile(r"(?:Section|Clause|Article|Rule|Chapter)\s+\d+(?:\.\d+)*", re.IGNORECASE),
    # Age/year quantities: "60 years", "30 days"
    re.compile(r"\b\d+\s+(?:years?|months?|days?|weeks?|hours?)\b", re.IGNORECASE),
    # Monetary amounts
    re.compile(r"Rs\.?\s*\d[\d,]*"),
    # Percentage
    re.compile(r"\b\d+(?:\.\d+)?%"),
]


def _extract_entities(text: str) -> list[str]:
    """
    Extract verifiable factual entities from text.

    Args:
        text: Answer or context string.

    Returns:
        Deduplicated list of entity strings.
    """
    found: set[str] = set()
    for pat in _ENTITY_PATTERNS:
        for m in pat.finditer(text):
            found.add(m.group().strip().lower())
    return sorted(found)


# ─────────────────────────────────────────────
# TOKEN UTILITIES
# ─────────────────────────────────────────────

_STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "shall",
    "should", "may", "might", "must", "can", "could", "of", "in", "on",
    "at", "to", "for", "with", "by", "from", "as", "or", "and", "but",
    "not", "that", "this", "it", "its", "which", "who", "whom", "their",
    "there", "they", "these", "those", "such", "any", "all", "also",
    "up", "into", "through", "after", "before", "about", "under", "above",
    "if", "so", "than", "then", "no", "yes",
}


def _content_tokens(text: str) -> set[str]:
    """Return the set of lowercase content words (stopwords removed)."""
    tokens = re.findall(r"\b\w+\b", text.lower())
    return {t for t in tokens if t not in _STOPWORDS and len(t) > 1}


def _token_overlap(answer: str, context: str) -> float:
    """
    Compute fraction of content tokens in `answer` that also appear in `context`.

    Args:
        answer:  Generated answer text.
        context: Combined text of all retrieved chunks.

    Returns:
        Overlap ratio (0.0 – 1.0).
    """
    answer_tokens  = _content_tokens(answer)
    context_tokens = _content_tokens(context)
    if not answer_tokens:
        return 0.0
    overlap = answer_tokens & context_tokens
    return len(overlap) / len(answer_tokens)


# ─────────────────────────────────────────────
# SENTENCE GROUNDING
# ─────────────────────────────────────────────

def _split_sentences(text: str) -> list[str]:
    """Split text into sentences."""
    sents = re.split(r"(?<=[.!?।])\s+", text)
    return [s.strip() for s in sents if s.strip() and len(s.split()) > 3]


def _sentence_coverage(answer: str, context: str) -> float:
    """
    Compute fraction of answer sentences that have at least partial support
    in the retrieved context (using token overlap ≥ 0.4 per sentence).

    Args:
        answer:  Generated answer.
        context: Combined context text.

    Returns:
        Coverage ratio (0.0 – 1.0).
    """
    sentences = _split_sentences(answer)
    if not sentences:
        return 1.0   # No verifiable sentences — pass by default

    supported = 0
    context_tokens = _content_tokens(context)

    for sent in sentences:
        sent_tokens = _content_tokens(sent)
        if not sent_tokens:
            supported += 1
            continue
        overlap = len(sent_tokens & context_tokens) / len(sent_tokens)
        if overlap >= 0.4:
            supported += 1
        else:
            logger.debug(
                "Low-support sentence (overlap=%.2f): %r", overlap, sent[:60]
            )

    return supported / len(sentences)


# ─────────────────────────────────────────────
# VERIFIER CLASS
# ─────────────────────────────────────────────

class AnswerVerifier:
    """
    Multi-stage verifier that checks whether a generated answer is grounded
    in the retrieved document chunks.

    Usage:
        verifier = AnswerVerifier()
        result   = verifier.verify(answer, results)
        if result.passed:
            print(result.answer)
        else:
            print(config.NOT_FOUND_RESPONSE)
    """

    def __init__(self) -> None:
        logger.info("AnswerVerifier initialised.")

    def verify(
        self,
        answer: str,
        results: list[RetrievalResult],
    ) -> VerificationResult:
        """
        Verify a generated answer against retrieved chunks.

        Args:
            answer:  Raw answer string from the RAG engine.
            results: RetrievalResult list used to generate the answer.

        Returns:
            VerificationResult with pass/fail decision, confidence, and citation.
        """
        with Timer("Answer verification", logger):
            return self._run_verification(answer, results)

    def _run_verification(
        self,
        answer: str,
        results: list[RetrievalResult],
    ) -> VerificationResult:
        """Internal verification logic."""

        # ── Trivial cases ──────────────────────────────────────────────────
        if not answer or not answer.strip():
            return self._fail("Answer is empty.", results)

        if config.NOT_FOUND_RESPONSE.lower() in answer.lower():
            # Model itself said not-found — accept gracefully
            logger.info("Model returned not-found response — treating as pass.")
            return VerificationResult(
                passed=True,
                answer=config.NOT_FOUND_RESPONSE,
                confidence=0.0,
                citation=self._build_citation(results),
            )

        if not results:
            return self._fail("No retrieved chunks to verify against.", results)

        # Combine context for comparisons
        context = " ".join(r.chunk.text for r in results)

        # ── Stage 1: Token overlap ─────────────────────────────────────────
        token_ov = _token_overlap(answer, context)
        logger.debug("Verification Stage 1 — token overlap: %.3f", token_ov)

        if token_ov < config.VERIFICATION_MIN_OVERLAP:
            return self._fail(
                f"Token overlap {token_ov:.2f} < threshold {config.VERIFICATION_MIN_OVERLAP}.",
                results,
                token_overlap=token_ov,
            )

        # ── Stage 2: Entity coverage ───────────────────────────────────────
        answer_entities  = _extract_entities(answer)
        context_entities = _extract_entities(context)
        context_lower    = context.lower()

        if answer_entities:
            supported_entities = [
                e for e in answer_entities
                if e in context_entities or e in context_lower
            ]
            entity_cov = len(supported_entities) / len(answer_entities)
        else:
            entity_cov = 1.0   # No extractable entities — neutral

        logger.debug(
            "Verification Stage 2 — entity coverage: %.3f "
            "(%d/%d entities supported)",
            entity_cov,
            len(supported_entities) if answer_entities else 0,
            len(answer_entities),
        )

        if entity_cov < 0.5 and answer_entities:
            unsupported = [
                e for e in answer_entities
                if e not in context_entities and e not in context_lower
            ]
            return self._fail(
                f"Entity coverage {entity_cov:.2f} < 0.5. "
                f"Unsupported: {unsupported[:3]}",
                results,
                token_overlap=token_ov,
                entity_coverage=entity_cov,
            )

        # ── Stage 3: Sentence grounding ────────────────────────────────────
        sent_cov = _sentence_coverage(answer, context)
        logger.debug("Verification Stage 3 — sentence coverage: %.3f", sent_cov)

        if sent_cov < 0.5:
            return self._fail(
                f"Sentence coverage {sent_cov:.2f} < 0.5.",
                results,
                token_overlap=token_ov,
                entity_coverage=entity_cov,
                sentence_coverage=sent_cov,
            )

        # ── Confidence computation ─────────────────────────────────────────
        confidence = (
            token_ov  * 40.0
            + entity_cov  * 40.0
            + sent_cov    * 20.0
        )
        # Bonus: very high top-1 similarity from retriever
        if results and results[0].score > 0.80:
            confidence = min(100.0, confidence + 5.0)

        logger.debug("Confidence score: %.2f", confidence)

        if confidence < config.VERIFICATION_MIN_CONFIDENCE:
            return self._fail(
                f"Confidence {confidence:.1f} < threshold "
                f"{config.VERIFICATION_MIN_CONFIDENCE}.",
                results,
                token_overlap=token_ov,
                entity_coverage=entity_cov,
                sentence_coverage=sent_cov,
            )

        # ── Build citation ─────────────────────────────────────────────────
        citation = self._build_citation(results)

        logger.info(
            "Verification PASSED — confidence=%.1f, token_overlap=%.3f, "
            "entity_cov=%.3f, sent_cov=%.3f",
            confidence, token_ov, entity_cov, sent_cov,
        )

        return VerificationResult(
            passed=True,
            answer=answer,
            confidence=round(confidence, 2),
            citation=citation,
            token_overlap=token_ov,
            entity_coverage=entity_cov,
            sentence_coverage=sent_cov,
        )

    # ── Helpers ──────────────────────────────────────────────────────────────

    @staticmethod
    def _fail(
        reason: str,
        results: list[RetrievalResult],
        token_overlap:    float = 0.0,
        entity_coverage:  float = 0.0,
        sentence_coverage: float = 0.0,
    ) -> VerificationResult:
        """Create a failed VerificationResult with diagnostics."""
        logger.warning("Verification FAILED — %s", reason)
        return VerificationResult(
            passed=False,
            answer=config.NOT_FOUND_RESPONSE,
            confidence=0.0,
            citation=AnswerVerifier._build_citation(results),
            token_overlap=token_overlap,
            entity_coverage=entity_coverage,
            sentence_coverage=sentence_coverage,
            failure_reason=reason,
        )

    @staticmethod
    def _build_citation(results: list[RetrievalResult]) -> CitationInfo:
        """
        Build CitationInfo from a list of retrieval results.

        The citation uses the highest-scoring chunk's section label and pages.
        """
        if not results:
            return CitationInfo()

        # Best chunk = highest score
        best = results[0]

        all_pages:  list[int] = []
        all_go:     list[str] = []
        all_chunks: list[int] = []
        all_scores: list[float] = []

        for r in results:
            all_chunks.append(r.chunk.chunk_id)
            all_scores.append(r.score)
            for p in r.chunk.page_numbers:
                if p not in all_pages:
                    all_pages.append(p)
            for go in r.chunk.metadata.get("go_numbers", []):
                if go not in all_go:
                    all_go.append(go)

        return CitationInfo(
            section_label     = best.chunk.section_label,
            page_numbers      = sorted(all_pages),
            go_numbers        = all_go,
            chunk_ids         = all_chunks,
            similarity_scores = all_scores,
        )


### `main.py`

In [ ]:
%%writefile main.py
"""
main.py
-------
Entry point for the AI Legal Question Answering System.

Run with:
    python main.py

Workflow per session:
    1. User uploads a document (PDF / DOCX / TXT).
    2. System extracts text (OCR if scanned).
    3. Language of the document is detected.
    4. Document text is translated to English if Tamil / Hindi.
    5. Text is cleaned (preprocessing).
    6. Document is chunked using structure-aware splitter.
    7. Embeddings are generated and stored in FAISS.
    8. Interactive Q&A loop begins:
         a. User types a question (English / Tamil / Hindi).
         b. Language detected; question translated to English.
         c. Relevant chunks retrieved via FAISS similarity search.
         d. Answer generated by RAG engine.
         e. Answer verified against retrieved context.
         f. Verified answer (or NOT_FOUND) + citation printed.
         g. Answer translated back to user's original language.
    9. User can ask another question or type 'exit' to quit.

No database. No web UI. Pure Python terminal application.
"""

import sys
import io
import logging
from pathlib import Path

# Force UTF-8 output and input on Windows terminals (fixes Unicode errors)
if sys.platform == "win32":
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")
    sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding="utf-8", errors="replace")
    sys.stdin  = io.TextIOWrapper(sys.stdin.buffer, encoding="utf-8", errors="replace")

# ── Ensure the project root is on sys.path ──────────────────────────────────
_PROJECT_ROOT = Path(__file__).parent.resolve()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import config
from utils import (
    setup_logging,
    validate_file,
    get_file_size_mb,
    cprint,
    print_separator,
    print_section_header,
    prompt_user,
    confirm,
    Timer,
)
from ocr          import OCRProcessor
from translator   import Translator
from preprocessor import TextPreprocessor
from chunker      import StructureAwareChunker
from embeddings   import EmbeddingModel
from retriever    import VectorRetriever
from rag_engine   import RAGEngine
from verification import AnswerVerifier

logger = setup_logging("main")


# ─────────────────────────────────────────────
# BANNER
# ─────────────────────────────────────────────

_BANNER = """
+======================================================================+
|                                                                      |
|     AI Legal & Government Document Question Answering System         |
|                                                                      |
|     Research Prototype  |  Tamil Nadu Government  |  Python 3.12+   |
|                                                                      |
|     Supported Languages : English  |  Tamil  |  Hindi               |
|     Supported Formats   : PDF  |  Scanned PDF  |  DOCX  |  TXT     |
|                                                                      |
|  [!] Answers are sourced ONLY from uploaded documents.               |
|      The system NEVER uses its own knowledge.                        |
|                                                                      |
+======================================================================+
"""


def print_banner() -> None:
    cprint(_BANNER, colour="cyan", bold=True)


# ─────────────────────────────────────────────
# OUTPUT FORMATTER
# ─────────────────────────────────────────────

def print_qa_result(
    original_question: str,
    detected_lang:     str,
    translated_question: str,
    retrieved_sections: list,
    answer_en:          str,
    answer_native:      str,
    verification,
) -> None:
    """
    Print the full QA result in the prescribed output format.
    """
    lang_name = config.SUPPORTED_LANGUAGES.get(detected_lang, detected_lang)

    print_separator("═", 70, "cyan")
    cprint("  QUESTION & ANSWER RESULT", colour="cyan", bold=True)
    print_separator("═", 70, "cyan")

    # Question metadata
    cprint(f"\n  Question         : {original_question}", colour="white", bold=True)
    cprint(f"  Detected Language: {lang_name} ({detected_lang})", colour="yellow")
    if detected_lang != "en":
        cprint(f"  Translated Q     : {translated_question}", colour="yellow")

    # Retrieved sections
    print_separator("─", 70)
    cprint("  Retrieved Sections:", colour="blue", bold=True)
    for r in retrieved_sections:
        label = r.chunk.section_label or f"Chunk {r.chunk.chunk_id}"
        pages = ", ".join(str(p) for p in r.chunk.page_numbers) or "N/A"
        cprint(
            f"    [{r.rank}] {label}  |  Page(s): {pages}  |  Score: {r.score_percent:.1f}%",
            colour="blue",
        )

    # Answer
    print_separator("─", 70)
    cprint("  Generated Answer:", colour="green", bold=True)
    if detected_lang != "en":
        cprint(f"  [English] : {answer_en}", colour="white")
        cprint(f"  [{lang_name}]: {answer_native}", colour="green", bold=True)
    else:
        cprint(f"  {answer_native}", colour="green", bold=True)

    # Verification
    print_separator("─", 70)
    v_colour = "green" if verification.passed else "red"
    cprint(
        f"  Verification     : {verification.status_label}",
        colour=v_colour,
        bold=True,
    )
    if not verification.passed and verification.failure_reason:
        cprint(f"  Failure Reason   : {verification.failure_reason}", colour="red")

    # Confidence & Citation
    if verification.passed and verification.confidence > 0:
        cprint(
            f"  Confidence Score : {verification.confidence_str}",
            colour="green",
        )
        cprint(
            f"  Source Citation  : {verification.citation.format()}",
            colour="yellow",
        )

    print_separator("═", 70, "cyan")
    print()


# ─────────────────────────────────────────────
# DOCUMENT LOADING PIPELINE
# ─────────────────────────────────────────────

class DocumentSession:
    """
    Encapsulates a single-document QA session.

    Attributes:
        file_path:   Path to the uploaded document.
        chunks:      List of DocumentChunk objects.
        retriever:   Populated VectorRetriever.
        doc_lang:    Detected document language.
    """

    def __init__(self) -> None:
        self.file_path  = None
        self.chunks     = []
        self.retriever  = None
        self.doc_lang   = "en"
        self.total_pages = 1

        # Shared component instances
        self._ocr         = OCRProcessor()
        self._translator  = Translator()
        self._preprocessor = TextPreprocessor()
        self._chunker     = StructureAwareChunker()
        self._embed_model = EmbeddingModel()
        self._rag         = RAGEngine()
        self._verifier    = AnswerVerifier()

    def load(self, file_path_str: str) -> None:
        """
        Full document loading pipeline:
        OCR → Language Detection → Translation → Preprocessing → Chunking → Indexing.

        Args:
            file_path_str: Path string provided by the user.
        """
        # ── Step 0: Validate file ────────────────────────────────────────────
        print_section_header("STEP 1 — Loading Document")
        file_path = validate_file(file_path_str)
        self.file_path = file_path
        size_mb = get_file_size_mb(file_path)
        cprint(f"  File : {file_path.name}", colour="white")
        cprint(f"  Size : {size_mb:.2f} MB", colour="white")

        # ── Step 1: OCR ──────────────────────────────────────────────────────
        cprint("\n  Extracting text (OCR if needed)…", colour="yellow")
        ocr_result = self._ocr.process(file_path)
        self.total_pages = ocr_result.total_pages
        cprint(
            f"  Extracted {ocr_result.total_pages} page(s). "
            f"OCR used: {ocr_result.used_ocr}",
            colour="green",
        )

        raw_text = ocr_result.raw_text
        if not raw_text.strip():
            raise ValueError("No text could be extracted from the document.")

        # ── Step 2: Language Detection ───────────────────────────────────────
        print_section_header("STEP 2 — Language Detection")
        self.doc_lang = self._translator.detect_language(raw_text)
        lang_name = config.SUPPORTED_LANGUAGES.get(self.doc_lang, "Unknown")
        cprint(
            f"  Document Language: {lang_name} ({self.doc_lang})",
            colour="cyan",
        )

        # ── Step 3: Translation (if not English) ─────────────────────────────
        if self.doc_lang != "en":
            print_section_header(f"STEP 3 — Translating {lang_name} → English")
            cprint(
                "  Translating document. This may take a few minutes for large files…",
                colour="yellow",
            )
            with Timer("Full document translation", logger):
                raw_text = self._translator.to_english(raw_text, self.doc_lang)
            cprint("  Translation complete.", colour="green")
        else:
            cprint("\n  Document is in English — translation skipped.", colour="dim")

        # ── Step 4: Preprocessing ────────────────────────────────────────────
        print_section_header("STEP 4 — Text Preprocessing")
        pre_result = self._preprocessor.clean(raw_text)
        cprint(
            f"  Cleaned {pre_result.original_length} → {pre_result.cleaned_length} chars "
            f"({pre_result.reduction_percent}% reduction).",
            colour="cyan",
        )
        if pre_result.go_numbers_found:
            cprint(
                f"  GO Numbers detected: {', '.join(pre_result.go_numbers_found)}",
                colour="yellow",
            )

        # ── Step 5: Chunking ─────────────────────────────────────────────────
        print_section_header("STEP 5 — Structure-Aware Chunking")
        self.chunks = self._chunker.chunk(
            pre_result.cleaned_text,
            pages=ocr_result.pages,
        )
        # Annotate chunks with GO numbers metadata
        for chunk in self.chunks:
            import re
            go_nums = re.findall(
                r"G\.O\.\s*(?:Ms\.|Rt\.|No\.?)?\s*\d+|GO\s*(?:No\.?)?\s*\d+",
                chunk.text,
                re.IGNORECASE,
            )
            chunk.metadata["go_numbers"] = go_nums

        cprint(f"  Created {len(self.chunks)} chunk(s).", colour="cyan")

        # ── Step 6: Embedding + Indexing ─────────────────────────────────────
        print_section_header("STEP 6 — Generating Embeddings & Indexing")
        cprint(
            "  Generating embeddings — first run downloads the model (~1-2 GB).",
            colour="yellow",
        )
        self.retriever = VectorRetriever(self._embed_model)
        self.retriever.index_chunks(self.chunks)
        cprint(
            f"  FAISS index built. {self.retriever.total_chunks} vectors in memory.",
            colour="green",
        )

        print_separator("═", 70, "green")
        cprint(
            "  ✓ Document ready for question answering!", colour="green", bold=True
        )
        print_separator("═", 70, "green")
        print()

    # ── Q&A ──────────────────────────────────────────────────────────────────

    def answer(self, question: str) -> None:
        """
        Process a user question end-to-end and print the result.

        Args:
            question: Raw question string in any supported language.
        """
        if not question.strip():
            cprint("  [!] Empty question — skipping.", colour="yellow")
            return

        # ── Detect question language ─────────────────────────────────────────
        question_lang = self._translator.detect_language(question)

        # ── Translate question to English ────────────────────────────────────
        question_en = self._translator.to_english(question, question_lang)
        logger.info(
            "Question | lang=%s | en=%r", question_lang, question_en[:100]
        )

        # ── Retrieve relevant chunks ─────────────────────────────────────────
        is_summary = any(w in question_en.lower() for w in ["summary", "summarise", "summarize"])
        if is_summary:
            results = self.retriever.retrieve(
                question_en,
                top_k=max(16, config.RETRIEVAL_TOP_K * 2),
                min_score=config.RETRIEVAL_MIN_SCORE * 0.5,
            )
        else:
            results = self.retriever.retrieve(question_en)
        if not results:
            cprint(
                "\n  No relevant sections found in the document.",
                colour="red",
            )
            self._print_not_found(question, question_lang)
            return

        # ── Generate answer (attempt 1) ──────────────────────────────────────
        answer_en = self._rag.generate(question_en, results, total_pages=self.total_pages)

        # ── Verify answer ────────────────────────────────────────────────────
        verification = self._verifier.verify(answer_en, results)

        # ── Retry if failed ──────────────────────────────────────────────────
        if not verification.passed:
            logger.info("Verification failed — retrying with wider retrieval.")
            results_wide = self.retriever.retrieve(
                question_en,
                top_k=config.RETRIEVAL_TOP_K + 3,
                min_score=config.RETRIEVAL_MIN_SCORE * 0.6,
            )
            answer_en_retry = self._rag.generate(question_en, results_wide, total_pages=self.total_pages)
            verification    = self._verifier.verify(answer_en_retry, results_wide)
            if verification.passed:
                answer_en = answer_en_retry
                results   = results_wide

        # ── Translate answer back to user's language ─────────────────────────
        answer_native = self._translator.from_english(
            verification.answer, question_lang
        )

        # ── Print result ─────────────────────────────────────────────────────
        print_qa_result(
            original_question   = question,
            detected_lang       = question_lang,
            translated_question = question_en,
            retrieved_sections  = results,
            answer_en           = verification.answer,
            answer_native       = answer_native,
            verification        = verification,
        )

    @staticmethod
    def _print_not_found(question: str, lang: str) -> None:
        """Print a minimal not-found result block."""
        lang_name = config.SUPPORTED_LANGUAGES.get(lang, lang)
        print_separator("═", 70, "red")
        cprint("  QUESTION & ANSWER RESULT", colour="red", bold=True)
        print_separator("═", 70, "red")
        cprint(f"\n  Question          : {question}", colour="white")
        cprint(f"  Detected Language : {lang_name}", colour="yellow")
        print_separator("─", 70)
        cprint(
            "  Generated Answer  : Information not found in the uploaded document.",
            colour="red",
            bold=True,
        )
        cprint("  Verification      : N/A (no matching context)", colour="red")
        print_separator("═", 70, "red")
        print()


# ─────────────────────────────────────────────
# INTERACTIVE LOOP
# ─────────────────────────────────────────────

def run_interactive(session: DocumentSession) -> None:
    """
    Main Q&A loop.
    Prompts the user for questions until they type 'exit' or 'quit'.
    """
    cprint(
        "  Type your question in English, Tamil, or Hindi.",
        colour="white",
    )
    cprint(
        "  Type 'exit' or 'quit' to end the session.",
        colour="dim",
    )
    print()

    while True:
        question = prompt_user("Your Question")
        if not question:
            continue
        if question.lower() in {"exit", "quit", "q", "bye"}:
            cprint("\n  Session ended. Goodbye!\n", colour="cyan")
            break
        session.answer(question)


# ─────────────────────────────────────────────
# MAIN ENTRY POINT
# ─────────────────────────────────────────────

def main() -> None:
    """
    Application entry point.

    Flow:
        1. Print banner.
        2. Prompt user for document path.
        3. Run document loading pipeline.
        4. Start interactive Q&A loop.
        5. Graceful shutdown on exit / Ctrl-C.
    """
    print_banner()

    session = DocumentSession()

    # ── Document upload ──────────────────────────────────────────────────────
    print_section_header("DOCUMENT UPLOAD")
    cprint(
        "  Supported: PDF, Scanned PDF, DOCX, TXT",
        colour="dim",
    )

    while True:
        path_str = prompt_user(
            "Enter the full path to your document (or 'exit' to quit)"
        )
        if path_str.lower() in {"exit", "quit"}:
            cprint("  Exiting. Goodbye!\n", colour="cyan")
            sys.exit(0)

        try:
            session.load(path_str)
            break
        except FileNotFoundError as exc:
            cprint(f"\n  [ERROR] {exc}", colour="red")
        except ValueError as exc:
            cprint(f"\n  [ERROR] {exc}", colour="red")
        except Exception as exc:
            logger.exception("Unexpected error during document loading.")
            cprint(f"\n  [ERROR] Unexpected error: {exc}", colour="red")
            cprint("  Please try a different document.", colour="yellow")

    # ── Q&A loop ─────────────────────────────────────────────────────────────
    try:
        run_interactive(session)
    except KeyboardInterrupt:
        cprint("\n\n  Interrupted by user. Goodbye!\n", colour="yellow")


if __name__ == "__main__":
    main()


## Cell 14 — Upload Your Document
Upload your PDF/DOCX/TXT file using the Colab file browser (left sidebar → 📁 icon),
then run the cell below. When prompted, enter the path shown after upload
(e.g. `/content/sample_tamil.pdf`).


In [ ]:
from google.colab import files
import os

print('Upload your document (PDF, DOCX, or TXT):')
uploaded = files.upload()
doc_path = '/content/' + list(uploaded.keys())[0]
print(f'\nDocument uploaded: {doc_path}')
os.environ['UPLOADED_DOC_PATH'] = doc_path


## Cell 15 — Run the QA System

In [ ]:
# Run the interactive QA system
# Type your questions when prompted. Type 'exit' to stop.
!python main.py
